### Loading the libraries

In [ ]:
import json
import os
import random
import re
import time
from datetime import datetime
from typing import Dict, List, Union, Tuple, Set, Optional

import gseapy as gp
import pandas as pd
import requests
from tqdm import tqdm

### Annotation Pipeline

In [ ]:
class OptimizedPathwayAnalyzer:
    def __init__(self):
        """Initialize the analyzer"""
        self.api_key = None
        self.api_url = "https://api.deepseek.com/v1/chat/completions"
        self.checkpoint_file = f"checkpoint_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        self.results_csv = f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        self.pathway_gene_mapping = {}  # Cache for pathway to gene mappings
        
    def set_api_key(self, api_key: str = None):
        """
        Set the Deepseek API key
        Prioritizes passed key, then environment variable
        """
        if api_key:
            self.api_key = api_key
        elif os.environ.get("DEEPSEEK_API_KEY"):
            self.api_key = os.environ.get("DEEPSEEK_API_KEY")
        else:
            raise ValueError("No Deepseek API key provided. Set it via argument or DEEPSEEK_API_KEY environment variable.")
    
    def _create_system_prompt(self) -> str:
        """Create the sophisticated system prompt"""
        return """ You are a bioinformatics expert. Focus on Lung Cancer. Provide detailed pathway analysis with scientific literature support.Always cite specific papers when discussing pathway relationships."""
    
    def _create_all_in_one_prompt(self, genes: List[str], enrichment_pathways: List[str]) -> str:
        """
        Create a unified prompt that handles pathway analysis types
        
        Args:
            genes: List of gene symbols
            enrichment_pathways: List of enrichment pathways
            
        Returns:
            Comprehensive system prompt for pathway analysis
        """
        enrichment_section = ""
        if enrichment_pathways and len(enrichment_pathways) > 0:
            enrichment_section = "\nEnrichment Analysis Results:\n" + "\n".join(f"- {pathway}" for pathway in enrichment_pathways)
        
        genes_str = ", ".join(genes)
        
        enrichment_instruction = "Using the enrichment pathways above (if any), provide an analysis of the gene set"
        
        return f"""You are a bioinformatics expert conducting pathway analysis of gene sets. For the gene set: [{genes_str}]{enrichment_section}

Please perform analysis in clearly labeled sections:

===== SECTION 1: ANALYSIS WITH ENRICHMENT =====
{enrichment_instruction}:
1. Propose a concise, descriptive name for the biological process
2. Assign a confidence score (0.00-1.00) for this process
3. Provide explicit reasoning for choosing this pathway/process
4. List the specific genes from the gene set that contribute to the process
5. IMPORTANT: A minimum of TWO genes from the gene set MUST be associated with a common biological process to annotate it as a valid pathway
6. If there isn't sufficient evidence or fewer than two genes are associated with a common pathway, label as "Unknown Pathway" with a low confidence score

Output format for this section:
PROCESS WITH ENRICHMENT: [Process Name] ([Confidence Score])

PATHWAY REASONING WITH ENRICHMENT:
[Your explicit reasoning for choosing this pathway/process]

CONTRIBUTING GENES WITH ENRICHMENT:
[Comma-separated list of contributing genes]

ANALYSIS TEXT WITH ENRICHMENT:
[Detailed analysis text explaining the biological significance and mechanisms of this process, without citations]

===== SECTION 2: ANALYSIS WITHOUT ENRICHMENT =====
Based SOLELY on your knowledge of these genes, without considering the enrichment results:
1. Propose a concise, descriptive name for the biological process
2. Assign a confidence score (0.00-1.00) for this process
3. List the specific genes from the gene set that are involved in the process
4. IMPORTANT: A minimum of TWO genes from the gene set MUST be associated with a common biological process to annotate it as a valid pathway
5. IMPORTANT: If there isn't sufficient evidence or fewer than two genes share a common biological function, label as "Unknown Pathway" with a low confidence score

Output format for this section:
PROCESS WITHOUT ENRICHMENT: [Process Name] ([Confidence Score])

CONTRIBUTING GENES WITHOUT ENRICHMENT:
[Comma-separated list of contributing genes]

PATHWAY REASONING WITHOUT ENRICHMENT:
[Your explicit reasoning for choosing this pathway/process]

ANALYSIS TEXT WITHOUT ENRICHMENT:
[Detailed analysis text explaining the biological significance and mechanisms of this process, without citations]

===== SECTION 3: FINAL PROCESS SELECTION =====
Compare both analyses and provide:
1. Which process (with or without enrichment) has higher confidence and why
2. Final reasoning for the chosen process

Output format for this section:
FINAL PROCESS REASONING:
[Detailed explanation of why you chose the final process, comparing both analyses and explaining which one provides stronger evidence]

Analytical Guidelines:
- Be concise and avoid unnecessary words
- Be factual without editorializing
- Be specific, avoiding overly general statements
- Avoid listing individual protein facts
- Group proteins by similar functions
- Discuss their interplay, synergistic or antagonistic effects
- Focus on functional integration within the system
- Include at least 2-3 specific paper citations when discussing established pathway relationships

Confidence Score Instructions:
- Assign a score from 0.00 to 1.00
- 0.00 indicates lowest confidence
- 1.00 reflects highest confidence
- Base the score on the proportion of genes participating in the identified process"""
    
    def _make_deepseek_request(self, prompt: str) -> str:
        """
        Make a request to the Deepseek API
        
        Args:
            prompt: The prompt to send to Deepseek
            
        Returns:
            The response text from Deepseek
        """
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        
        data = {
            "model": "deepseek-chat",
            "messages": [
                {"role": "system", "content": self._create_system_prompt()},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0,
            "max_tokens": 4000
        }
        
        response = requests.post(self.api_url, headers=headers, json=data)
        
        if response.status_code == 200:
            result = response.json()
            return result['choices'][0]['message']['content']
        else:
            raise Exception(f"Deepseek API request failed: {response.status_code} - {response.text}")
    
    def parse_communities(self, input_text: str) -> Dict[str, List[str]]:
        """Parse the input text into a dictionary of community numbers and gene lists."""
        communities = {}
        for line in input_text.strip().split('\n'):
            line = line.strip()
            if 'Refined Community' in line:
                match = re.search(r'Refined Community (\d+): \[(.*?)\]', line)
                if match:
                    community_num = match.group(1)
                    genes_str = match.group(2)
                    genes = [g.strip("'") for g in genes_str.split(', ')]
                    communities[community_num] = genes
        return communities
    
    def perform_enrichment(self, gene_list: List[str]) -> List[str]:

        try:
            databases = ['GO_Biological_Process_2021', 'Reactome_2022', 'KEGG_2021_Human']
            #databases = ['GO_Biological_Process_2025',
            #'GO_Molecular_Function_2025',
            #'GO_Cellular_Component_2025',
            #'Reactome_Pathways_2024',
            #'KEGG_2021_Human',
            #WikiPathways_2024_Human',
            #'BioPlanet_2019',
            #'MSigDB_Hallmark_2020',
            #'PFOCR_Pathways_2023',]
            
            all_results = []
            for database in databases:
                enr = gp.enrichr(gene_list=gene_list,
                                gene_sets=[database],
                                organism='Human',
                                outdir=None,
                                no_plot=True,
                                cutoff=0.01)
                
                results_df = enr.results
                if not results_df.empty:
                    results_df = results_df.sort_values('Adjusted P-value')
                    # Select top results
                    for _, row in results_df.head(5).iterrows():
                        all_results.append(row['Term'])
            
            # Return unique pathways
            return list(set(all_results))
        
        except Exception as e:
            print(f"Enrichment analysis failed: {str(e)}")
            return []
    
    def _clean_markdown_formatting(self, text: str) -> str:

        if not text or not isinstance(text, str):
            return text
            
        text = re.sub(r'\*([^*]+)\*', r'\1', text)
        
        text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
        
        text = re.sub(r'(?<!\w)\*(?!\w)', '', text)
        
        return text.strip()
    
    def _extract_section(self, text: str, section_name: str) -> str:

        patterns = [
            rf"{re.escape(section_name)}:\s*(.*?)(?=\n\n[\w\s]+:|===|$)",  # Main pattern
            rf"{re.escape(section_name)}:\s*(.*?)(?=\n[A-Z][A-Z\s]+:|$)",
            rf"{re.escape(section_name)}:\s*(.*?)(?=\n\*\*|$)" 
        ]
        
        for pattern in patterns:
            match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
            if match:
                result = match.group(1).strip()
                if result:
                    return self._clean_markdown_formatting(result)
        
        return f"{section_name} not found"
    
    def _extract_process_info(self, text: str, prefix: str) -> Tuple[str, float]:

        patterns = [
            rf"PROCESS {prefix} ENRICHMENT:\s*([^(]+?)\s*\(([0-9.]+)\)",
            rf"PROCESS {prefix} ENRICHMENT:\s*([^(]*?)\s*\(\s*([0-9.]+)\s*\)",
            rf"PROCESS {prefix} ENRICHMENT:\s*(.*?)\s*\(\s*Confidence Score:\s*([0-9.]+)\s*\)",
            rf"PROCESS {prefix} ENRICHMENT:\s*(.*?)\s*\*\*Confidence Score:\s*([0-9.]+)\*\*",
            rf"PROCESS {prefix} ENRICHMENT:\s*(.*?)\s*\(\s*\*\*Confidence Score:\s*([0-9.]+)\*\*\s*\)"
        ]
        
        for pattern in patterns:
            match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
            if match:
                process_name = match.group(1).strip()
                try:
                    confidence_score = float(match.group(2))
                    process_name = self._clean_markdown_formatting(process_name)
                    process_name = process_name.strip()
                    if process_name and confidence_score >= 0:
                        return process_name, confidence_score
                except ValueError:
                    continue
        
        simple_pattern = rf"PROCESS {prefix} ENRICHMENT:\s*(.*?)(?=\n|$)"
        match = re.search(simple_pattern, text, re.IGNORECASE)
        if match:
            process_name = match.group(1).strip()
            process_name = self._clean_markdown_formatting(process_name)
            process_name = re.sub(r'\s*\([^)]*$', '', process_name)
            process_name = process_name.strip()
            return process_name, 0.0
        
        return f"Unknown Process {prefix} Enrichment", 0.0
    
    def _extract_final_process(self, text: str) -> str:

        match = re.search(r"FINAL PROCESS REASONING:(.*?)(?=\n\n|$)", text, re.DOTALL | re.IGNORECASE)
        if match:
            result = match.group(1).strip()
            return self._clean_markdown_formatting(result)
        return "Final process reasoning not found"
    
    def analyze_community_optimized(self, comm_id: str, genes: List[str]) -> Dict:

        results = {
            "Community": comm_id,
            "Genes": genes,
            "Genes_String": ", ".join(genes)
        }
        
        try:
            enrichment_pathways = self.perform_enrichment(genes)
            results["Enrichment_Pathways"] = enrichment_pathways
            
            all_in_one_prompt = self._create_all_in_one_prompt(genes, enrichment_pathways)
            
            full_analysis = self._make_deepseek_request(all_in_one_prompt)
            results["Full_Analysis"] = full_analysis

            process_with_enrichment, confidence_with_enrichment = self._extract_process_info(full_analysis, "WITH")
            results["Process_With_Enrichment"] = process_with_enrichment
            results["Confidence_With_Enrichment"] = confidence_with_enrichment

            process_without_enrichment, confidence_without_enrichment = self._extract_process_info(full_analysis, "WITHOUT")
            results["Process_Without_Enrichment"] = process_without_enrichment
            results["Confidence_Without_Enrichment"] = confidence_without_enrichment
            
            try:
                results["Pathway_Reasoning_With_Enrichment"] = self._extract_section(full_analysis, "PATHWAY REASONING WITH ENRICHMENT")
            except Exception as e:
                results["Pathway_Reasoning_With_Enrichment"] = f"Extraction failed: {str(e)}"
                
            try:
                results["Pathway_Reasoning_Without_Enrichment"] = self._extract_section(full_analysis, "PATHWAY REASONING WITHOUT ENRICHMENT")
            except Exception as e:
                results["Pathway_Reasoning_Without_Enrichment"] = f"Extraction failed: {str(e)}"
            
            try:
                results["Contributing_Genes_With_Enrichment"] = self._extract_section(full_analysis, "CONTRIBUTING GENES WITH ENRICHMENT")
            except Exception as e:
                results["Contributing_Genes_With_Enrichment"] = f"Extraction failed: {str(e)}"
                
            try:
                results["Contributing_Genes_Without_Enrichment"] = self._extract_section(full_analysis, "CONTRIBUTING GENES WITHOUT ENRICHMENT")
            except Exception as e:
                results["Contributing_Genes_Without_Enrichment"] = f"Extraction failed: {str(e)}"
            
            try:
                results["Analysis_Text_With_Enrichment"] = self._extract_section(full_analysis, "ANALYSIS TEXT WITH ENRICHMENT")
            except Exception as e:
                results["Analysis_Text_With_Enrichment"] = f"Extraction failed: {str(e)}"
                
            try:
                results["Analysis_Text_Without_Enrichment"] = self._extract_section(full_analysis, "ANALYSIS TEXT WITHOUT ENRICHMENT")
            except Exception as e:
                results["Analysis_Text_Without_Enrichment"] = f"Extraction failed: {str(e)}"
            
            try:
                final_reasoning = self._extract_section(full_analysis, "FINAL PROCESS REASONING")
                results["Final_Process_Reasoning"] = final_reasoning
            except Exception as e:
                results["Final_Process_Reasoning"] = f"Extraction failed: {str(e)}"
            
            if confidence_with_enrichment >= confidence_without_enrichment:
                results["Final_Process"] = process_with_enrichment
                results["Final_Confidence"] = confidence_with_enrichment
                results["Final_Contributing_Genes"] = results.get("Contributing_Genes_With_Enrichment", "Not available")
            else:
                results["Final_Process"] = process_without_enrichment
                results["Final_Confidence"] = confidence_without_enrichment
                results["Final_Contributing_Genes"] = results.get("Contributing_Genes_Without_Enrichment", "Not available")
            
            return results
            
        except Exception as e:
            error_msg = f"Analysis failed: {str(e)}"
            print(f"Error analyzing community {comm_id}: {e}")
            results["Error"] = error_msg
            results.update({
                "Process_With_Enrichment": f"Error: {error_msg}",
                "Confidence_With_Enrichment": 0.0,
                "Process_Without_Enrichment": f"Error: {error_msg}",
                "Confidence_Without_Enrichment": 0.0,
                "Final_Process": f"Error: {error_msg}",
                "Final_Confidence": 0.0,
                "Full_Analysis": f"Error: {error_msg}",
                "Pathway_Reasoning_With_Enrichment": "Error occurred",
                "Pathway_Reasoning_Without_Enrichment": "Error occurred",
                "Contributing_Genes_With_Enrichment": "Error occurred",
                "Contributing_Genes_Without_Enrichment": "Error occurred",
                "Analysis_Text_With_Enrichment": "Error occurred",
                "Analysis_Text_Without_Enrichment": "Error occurred",
                "Final_Process_Reasoning": "Error occurred",
                "Final_Contributing_Genes": "Error occurred"
            })
            return results
    
    def analyze_communities_batch(self, communities: Dict[str, List[str]], batch_size: int = 1) -> List[Dict]:

        all_results = []
        community_items = list(communities.items())
        
        from tqdm import tqdm
        
        progress_bar = tqdm(total=len(community_items), desc="Analyzing communities", unit="community")
        
        for i in range(0, len(community_items), batch_size):
            batch = community_items[i:i+batch_size]
            
            for comm_id, genes in batch:
                result = self.analyze_community_optimized(comm_id, genes)
                all_results.append(result)

                progress_bar.update(1)

                processed = len(all_results)
                total = len(community_items)
                progress_bar.set_description(f"Processed {processed}/{total} communities")
                
                progress_bar.set_postfix({"Current": f"Community {comm_id}", "Genes": len(genes)})
        
        progress_bar.close()
        
        return all_results
    
    def create_detailed_dataframe(self, communities_text: str, batch_size: int = 1) -> pd.DataFrame:

        communities = self.parse_communities(communities_text)
        all_results = self.analyze_communities_batch(communities, batch_size)
        
        full_df = pd.DataFrame(all_results)
        
        requested_columns = [
            "Community",
            "Genes_String",
            "Enrichment_Pathways",
            "Process_With_Enrichment",
            "Confidence_With_Enrichment", 
            "Pathway_Reasoning_With_Enrichment",
            "Contributing_Genes_With_Enrichment",
            "Process_Without_Enrichment",
            "Confidence_Without_Enrichment",
            "Pathway_Reasoning_Without_Enrichment",
            "Contributing_Genes_Without_Enrichment", 
            "Final_Process",
            "Final_Confidence",
            "Final_Contributing_Genes",
            "Final_Process_Reasoning",
            "Full_Analysis"
        ]
        
        available_columns = [col for col in requested_columns if col in full_df.columns]
        return full_df[available_columns]

def run_optimized_analysis(input_text: str, api_key: str = None, detailed_csv: str = None, batch_size: int = 1) -> pd.DataFrame:

    analyzer = OptimizedPathwayAnalyzer()
    analyzer.set_api_key(api_key)
    
    detailed_df = analyzer.create_detailed_dataframe(input_text, batch_size)
    
    if detailed_csv:
        detailed_df.to_csv(detailed_csv, index=False)
    
    return detailed_df

if __name__ == "__main__":
    input_text = """
        Refined Community 1: ['ACTB', 'ACTG1', 'ANXA1', 'ANXA8', 'ATP5F1D', 'CALU', 'EPHX1', 'FGF4', 'GFAP', 'GRK4', 'GSTP1', 'HPRT1', 'HSPA5', 'KRT1', 'KRT19', 'KRT7', 'KRT8', 'MYCN', 'P4HB', 'PDIA5', 'PEBP1', 'PGAM1', 'PGK1', 'PKM', 'PLCB3', 'RAB14', 'SERPINA1', 'SOD2']
Refined Community 2: ['ACVRL1', 'DDR1', 'FGFR4', 'INSRR', 'IRS1', 'LMTK3', 'LRP1B', 'MAST4', 'MYB', 'NTRK2', 'PDGFRA', 'PIK3C2G', 'PRKDC', 'PTPRD', 'PTPRG', 'ROBO2', 'SMG1', 'TP53', 'TYK2', 'VAV1']
Refined Community 3: ['AKAP9', 'ANKRD10', 'ANKRD46', 'ARMC1', 'ARMC8', 'ASH2L', 'ATP11B', 'BAG4', 'BRMS1L', 'BRWD1', 'C19orf12', 'CBLL1', 'CDC42SE2', 'CDK8', 'CEBPG', 'CHD7', 'CMAS', 'COG3', 'COG5', 'CTTN', 'DBR1', 'DCAF13', 'DCTD', 'DNAJC2', 'E2F3', 'EID2', 'EIF3H', 'FAM91A1', 'FBXO25', 'FBXO3', 'FBXO33', 'FBXW11', 'FIG4', 'FXR1', 'GABPA', 'GATAD1', 'GEMIN2', 'GOLGA7', 'GTF3A', 'GZF1', 'ING1', 'IPO8', 'KBTBD11', 'KPNA4', 'KRIT1', 'LSM1', 'MACROH2A1', 'MAP3K1', 'MAPK8', 'METTL4', 'MFN1', 'MIPEP', 'MRPL47', 'MRPS11', 'MRPS28', 'MTMR6', 'MYNN', 'NAPG', 'NDUFB5', 'NEK3', 'NIPBL', 'NSD3', 'NUP153', 'PAK2', 'PAN3', 'PDCD6', 'PEPD', 'PHC3', 'PIK3CA', 'PNPT1', 'PPFIA1', 'PPM1A', 'PPP1R12A', 'PPP4R2', 'PRKCI', 'RAB20', 'RAE1', 'RASA1', 'RB1CC1', 'RFC4', 'RYK', 'SDHC', 'SEC23IP', 'SENP2', 'SIAH2', 'SNAP29', 'TAF2', 'TBCA', 'TERF1', 'THOC1', 'THUMPD1', 'TOMM70', 'TPD52', 'TRIM35', 'TXNL1', 'URI1', 'VPS41', 'VPS8', 'WASHC5', 'ZNF302']
Refined Community 4: ['ACVR1B', 'APC', 'BAP1', 'FYN', 'IRAK2', 'LTK', 'NF1', 'NOTCH4', 'PIK3R2', 'RB1', 'STK11', 'TP53']
Refined Community 5: ['EGFR', 'FLT4', 'KRAS', 'NRAS', 'STK11', 'TP53']
Refined Community 6: ['APC', 'ATM', 'CDKN2A', 'EGFR', 'EPHA3', 'EPHA5', 'ERBB4', 'FGFR4', 'GNAS', 'INHBA', 'KDR', 'KRAS', 'LRP1B', 'LTK', 'NF1', 'NRAS', 'NTRK1', 'NTRK3', 'PAK3', 'PDGFRA', 'PTPRD', 'RB1', 'SLC38A3', 'STK11', 'TP53', 'ZMYND10']
Refined Community 7: ['ACTR2', 'ADAMTS1', 'ADGRF1', 'AGTR2', 'AHRR', 'ANK2', 'APOL6', 'AQP4', 'ATP6V0D2', 'AZGP1', 'B3GNT5', 'BTG2', 'C1S', 'CAMKK2', 'CCL20', 'CRTC1', 'CTNNA1', 'CYP1A1', 'CYTH1', 'DDX3X', 'DUSP1', 'EIF1', 'ENG', 'EPAS1', 'EZR', 'FER', 'FGG', 'FKBP5', 'FOLH1', 'FUS', 'GABPB2', 'HAS1', 'HOXC8', 'HSP90AB1', 'HSPD1', 'IGLJ3', 'IL6ST', 'IQGAP1', 'ITGB6', 'ITLN1', 'MAPKAPK2', 'MAT2A', 'MATR3', 'MCL1', 'MET', 'MICALL2', 'MMP1', 'MVD', 'MYH11', 'NAMPT', 'NEAT1', 'NFASC', 'NR4A3', 'OTUD4', 'PGS1', 'PRKG1', 'RAB12', 'RASEF', 'RCAN1', 'RHOB', 'RNF125', 'RNF144B', 'SCAF11', 'SEC61A1', 'SERPINB2', 'SERPINE1', 'SOD2', 'SRPRA', 'SRSF5', 'STC1', 'STEEP1', 'SUPT16H', 'TAOK1', 'TOP1', 'TTC3', 'UFM1', 'USP10', 'WDR1', 'XAGE1A', 'ZFP36L2', 'ZNF649']
Refined Community 8: ['BAHCC1', 'BEST2', 'CACNG7', 'CCER1', 'CFAP65', 'DLGAP4', 'ELF3', 'FAM135B', 'FAM222B', 'FRMD4A', 'HNRNPA3P1', 'IGFLR1', 'INSL6', 'JAK3', 'KIF2B', 'LAMA3', 'MAGEB6', 'MEFV', 'MELK', 'MKRN3', 'NGEF', 'NLRP10', 'NLRP11', 'NLRP8', 'NTF3', 'PRAME', 'RBBP8NL', 'RNF175', 'SCUBE1', 'SHISA7', 'SOHLH2', 'TCL1B', 'TRIM29', 'ZNF556', 'ZNF572']
Refined Community 9: ['ABCG4', 'ACOT12', 'ACP1', 'ACTB', 'ADAM30', 'ADAMTS17', 'ADAMTSL4', 'ADGRD2', 'ADIPOR2', 'ADIRF', 'ADRA1A', 'AGTR1', 'AHI1', 'AJAP1', 'ALDOC', 'ALX1', 'AMIGO3', 'AMT', 'ANGPTL2', 'ANK2', 'ANKLE1', 'ANKS1B', 'AOX1', 'APCDD1L', 'ARHGAP22', 'ARHGEF1', 'ARRDC3', 'ATP2A2', 'ATP8A2', 'BANP', 'BCAS2', 'BCAT1', 'BCOR', 'BEGAIN', 'BNC1', 'BOD1', 'BRD9', 'BTG3', 'C1QTNF1', 'C3orf62', 'CA7', 'CACNA1C', 'CACNA2D2', 'CADPS', 'CANX', 'CARD9', 'CAVIN1', 'CCDC140', 'CCDC62', 'CCDC86', 'CCDC87', 'CD300A', 'CD81', 'CD93', 'CDH13', 'CFAP410', 'CIDEB', 'CLEC1A', 'CLMP', 'CLXN', 'CMTM2', 'CNBP', 'CNIH3', 'CNN1', 'CNNM1', 'CNTNAP5', 'COG1', 'COL1A1', 'COL1A2', 'COL2A1', 'COL5A1', 'COPZ2', 'CPPED1', 'CPT1B', 'CSF1', 'CSMD1', 'CST6', 'CSTF2T', 'CXCL12', 'CYP27A1', 'CYP2W1', 'CYYR1', 'DAB2IP', 'DAZAP1', 'DIO3OS', 'DIPK2B', 'DLC1', 'DOK7', 'DPEP3', 'DPH7', 'DRD4', 'DTNB', 'DUOX1', 'DYDC2', 'DYRK1B', 'DYSF', 'ECHDC1', 'EFCAB6', 'EIF2D', 'EIF4EBP3', 'ELAVL2', 'ENPP2', 'EPHA5', 'ESR1', 'ESX1', 'ETV7', 'EXOC6', 'EXOC7', 'FAF1', 'FAM107A', 'FAM83F', 'FBLIM1', 'FBP1', 'FBXO39', 'FECH', 'FERD3L', 'FGF11', 'FGF4', 'FRZB', 'FUCA2', 'GABRA2', 'GABRQ', 'GAL3ST1', 'GALNT15', 'GATA3', 'GBGT1', 'GCKR', 'GDI1', 'GFOD1', 'GFPT2', 'GIMAP8', 'GIPC2', 'GNA14', 'GP1BA', 'GRID2', 'GRIN2A', 'GRK2', 'GRM7', 'GTF2A1L', 'HAAO', 'HCN4', 'HDAC7', 'HLX', 'HNF1B', 'HOXA13', 'HOXA7', 'HOXA9', 'HOXB4', 'HOXC12', 'HOXD12', 'HOXD13', 'HOXD3', 'HPDL', 'HPN', 'HSPB6', 'HSPB9', 'HTR5A', 'ID4', 'IGF2-AS', 'IGFBP4', 'IL16', 'INSRR', 'IQCA1', 'IRF4', 'ISL1', 'ISYNA1', 'ITPKB', 'IZUMO1', 'JPH1', 'JPH2', 'KCNA4', 'KCNC4', 'KCNH7', 'KCNJ3', 'KDM1A', 'KIF12', 'KIT', 'KL', 'KRT72', 'L1TD1', 'LCK', 'LDLRAD4', 'LDOC1', 'LHPP', 'LHX1', 'LIMD2', 'LIN28A', 'LRFN5', 'LRIG1', 'LRRC34', 'LRRC3B', 'LRRC61', 'LRRC71', 'LSP1', 'LXN', 'LYL1', 'MAL', 'MAP4K3', 'MARVELD1', 'ME3', 'MEDAG', 'MEPCE', 'MINDY3', 'MLH3', 'MMRN2', 'MOS', 'MRAP2', 'MT1E', 'NAGS', 'NCL', 'NECAB1', 'NEFL', 'NEFM', 'NELL1', 'NFAM1', 'NFIX', 'NHLH2', 'NID2', 'NKAIN4', 'NLRC3', 'NME4', 'NOS3', 'OAZ2', 'OLFM1', 'OSM', 'OTOF', 'OTOP2', 'P2RY11', 'PAM', 'PAPOLB', 'PARP6', 'PAX1', 'PAX9', 'PCDHGA3', 'PCDHGB2', 'PCDHGC4', 'PDE4DIP', 'PDE8B', 'PDLIM1', 'PENK', 'PHF3', 'PHKG1', 'PIK3R5', 'PKP1', 'PLCB2', 'PLEKHF2', 'PLTP', 'PNMA1', 'PNMT', 'POLH', 'POU2AF1', 'POU4F2', 'PRDM8', 'PRLHR', 'PROM1', 'PROP1', 'PRRC1', 'PRSS50', 'PRTN3', 'PRXL2A', 'PYGO1', 'RALBP1', 'RAPGEF4', 'RAPGEFL1', 'RARB', 'RARRES2', 'RASGRP1', 'RASGRP2', 'RBBP6', 'RCSD1', 'REC8', 'RETREG1', 'RGMA', 'RGN', 'RIC3', 'RINL', 'RNF152', 'RSPH6A', 'RSPO2', 'RYR3', 'SARM1', 'SCARF2', 'SEC31B', 'SECTM1', 'SH3D21', 'SH3YL1', 'SLAMF1', 'SLC17A6', 'SLC22A3', 'SLC26A9', 'SLC27A6', 'SLC29A2', 'SLC35C2', 'SLC46A3', 'SLC47A1', 'SLC49A3', 'SLC4A11', 'SLC5A7', 'SLC6A2', 'SLCO4A1', 'SLIT2', 'SLITRK4', 'SNX33', 'SNX6', 'SOWAHA', 'SOX1', 'SPAG6', 'SPESP1', 'SPOCK2', 'ST6GAL2', 'ST6GALNAC3', 'STK32B', 'SUFU', 'SULF2', 'SVIL', 'SYNGR3', 'SYT10', 'TACR1', 'TACR3', 'TAGLN', 'TBX20', 'TBXT', 'TCF21', 'TEX26', 'TF', 'TFAP2A', 'TGFB1I1', 'TGFBI', 'TLL2', 'TMED2', 'TMEM101', 'TMEM150A', 'TMEM179B', 'TMEM204', 'TMEM74B', 'TMPRSS12', 'TMPRSS6', 'TMSB10', 'TNFRSF10C', 'TPPP3', 'TRAF5', 'TRIP6', 'TRPV4', 'TSHR', 'TWIST1', 'TYMP', 'UCN', 'UHRF1', 'UNKL', 'UPB1', 'VAMP5', 'VPREB1', 'VPS9D1', 'VWC2', 'VWCE', 'WT1-AS', 'XPO5', 'YPEL3', 'ZBTB47', 'ZCCHC24', 'ZDHHC11', 'ZFP69', 'ZFP69B', 'ZIM2', 'ZMYM6', 'ZNF154', 'ZNF177', 'ZNF22', 'ZNF22-AS1', 'ZNF428', 'ZNF454', 'ZNF513', 'ZNF534', 'ZNF626', 'ZNF671', 'ZNF785', 'ZNF875']
Refined Community 10: ['ABCA8', 'ADAMTS8', 'ADAMTSL3', 'ADARB1', 'AGER', 'CAV1', 'MCL1', 'NR2F2', 'PGM5', 'PHACTR2', 'RASL12', 'SSC4D', 'VWF']
Refined Community 11: ['CD47', 'CIT', 'CPSF2', 'CS', 'DHFR', 'MCU', 'MEAF6', 'P4HA1', 'P4HTM', 'PA2G4', 'PATL1', 'PRORP', 'SMNDC1', 'WAPL']
Refined Community 12: ['CDCA5', 'DSC3', 'DSG3', 'KRT14', 'KRT5', 'NECTIN1', 'OSGEP', 'PSMA8', 'SELENOI', 'SFN', 'SLC2A1', 'TOP2A', 'TPX2']
Refined Community 13: ['AKT1', 'AKT2', 'AKT3', 'ARAF', 'BAD', 'BRAF', 'CASP9', 'CCND1', 'CDK4', 'CDK6', 'CDKN2A', 'E2F1', 'E2F2', 'E2F3', 'EGF', 'EGFR', 'ERBB2', 'FHIT', 'FOXO3', 'GRB2', 'HRAS', 'KRAS', 'MAP2K1', 'MAP2K2', 'MAPK1', 'MAPK3', 'NRAS', 'PDPK1', 'PIK3CA', 'PIK3CB', 'PIK3CD', 'PIK3CG', 'PIK3R1', 'PIK3R2', 'PIK3R3', 'PIK3R5', 'PLCG1', 'PLCG2', 'PRKCA', 'PRKCB', 'PRKCG', 'RAF1', 'RARB', 'RASSF1', 'RASSF5', 'RB1', 'RXRA', 'RXRB', 'RXRG', 'SOS1', 'SOS2', 'STK4', 'TGFA', 'TP53']
Refined Community 14: ['AKT1', 'AKT2', 'AKT3', 'APAF1', 'BCL2', 'BCL2L1', 'BIRC2', 'BIRC3', 'CASP9', 'CCND1', 'CCNE1', 'CCNE2', 'CDK2', 'CDK4', 'CDK6', 'CDKN1B', 'CDKN2B', 'CHUK', 'CKS1B', 'COL4A1', 'COL4A2', 'COL4A4', 'COL4A6', 'CYCS', 'E2F1', 'E2F2', 'E2F3', 'FHIT', 'FN1', 'IKBKB', 'IKBKG', 'ITGA2', 'ITGA2B', 'ITGA3', 'ITGA6', 'ITGAV', 'ITGB1', 'LAMA1', 'LAMA2', 'LAMA3', 'LAMA4', 'LAMA5', 'LAMB1', 'LAMB2', 'LAMB3', 'LAMB4', 'LAMC1', 'LAMC2', 'LAMC3', 'MAX', 'MYC', 'NFKB1', 'NFKBIA', 'NOS2', 'PIAS1', 'PIAS2', 'PIAS3', 'PIAS4', 'PIK3CA', 'PIK3CB', 'PIK3CD', 'PIK3CG', 'PIK3R1', 'PIK3R2', 'PIK3R3', 'PIK3R5', 'PTEN', 'PTGS2', 'PTK2', 'RARB', 'RB1', 'RELA', 'RXRA', 'RXRB', 'RXRG', 'SKP2', 'TP53', 'TRAF1', 'TRAF2', 'TRAF3', 'TRAF4', 'TRAF5', 'TRAF6', 'XIAP']
Refined Community 15: ['ABCC3', 'ADGRG1', 'AKR1A1', 'AKR1B10', 'AKR1C1', 'ALDOA', 'APRT', 'ARL6IP1', 'ASCL1', 'ATP1B1', 'ATP5MF', 'BGN', 'CALR', 'CANX', 'CBX3', 'CCT3', 'CCT5', 'CD46', 'CDH1', 'CDK2AP2', 'CEACAM5', 'CEACAM6', 'CHIT1', 'CKAP4', 'CLDN4', 'CLDN9', 'COL11A1', 'COL1A1', 'COL1A2', 'COX6A1', 'CRABP2', 'CRH', 'CSTB', 'CYCS', 'DAP', 'DDOST', 'DENND2B', 'DPP7', 'DSP', 'EEF1A2', 'EEF1D', 'EIF3C', 'ELOC', 'ENC1', 'ENO1', 'ERBB2', 'ERBB3', 'FGB', 'FGG', 'FSCN1', 'GALNS', 'GFUS', 'GIPR', 'GPI', 'HAX1', 'HDGF', 'HPN', 'HSP90B1', 'HSPA5', 'HSPB1', 'HSPD1', 'HSPE1', 'HYOU1', 'IDH2', 'IGFBP2', 'IGFBP3', 'IGKC', 'JTB', 'JUP', 'KDELR1', 'KPNA4', 'KRT5', 'KRT6C', 'KRT7', 'LAPTM4B', 'LCN2', 'LDHA', 'LDHB', 'LGALS3BP', 'LTF', 'MAGED1', 'MALL', 'MARCKSL1', 'MCL1', 'MDK', 'MIF', 'MMP12', 'MSLN', 'MUC1', 'NACA', 'NME1', 'NME2', 'NME4', 'NQO1', 'NTS', 'ODC1', 'P4HB', 'PABPC1', 'PDCD6', 'PFKP', 'PFN2', 'PGD', 'PHLDA2', 'PKM', 'PLAU', 'PLP2', 'PLPP2', 'POSTN', 'PRDX1', 'PRDX4', 'PSMB4', 'PSMD8', 'PTPRF', 'RAB1A', 'RAN', 'RPL10A', 'RPL13', 'RPL17', 'RPL23A', 'RPL27', 'RPL27A', 'RPL28', 'RPL30', 'RPL31', 'RPL35', 'RPL36A', 'RPL37', 'RPL37A', 'RPL38', 'RPL7', 'RPL8', 'RPS10', 'RPS16', 'RPS2', 'RPS20', 'RPS21', 'RPS27', 'RPS4X', 'RPS5', 'RPS7', 'S100P', 'SCNN1A', 'SDHA', 'SFN', 'SLC22A18', 'SLC7A5', 'SNRPD2', 'SNRPE', 'SOX4', 'SPINK1', 'SPINT1', 'SPINT2', 'SPP1', 'SSR4', 'STAT1', 'SUMO2', 'SYN1', 'SYNGR2', 'TALDO1', 'TFF1', 'TFF3', 'THBS2', 'TIMP1', 'TKT', 'TOP1', 'TRAF4', 'TRIM2', 'TRIM28', 'TSPAN8', 'TUFM', 'TXNRD1', 'UBE2C', 'UBE2I', 'UCHL1', 'UGDH', 'UQCRB', 'UQCRH', 'VEGFA', 'XBP1', 'YIF1A', 'YWHAQ', 'YWHAZ', 'ZNF146']
Refined Community 16: ['ALDH1A1', 'CALR', 'CCNE1', 'CDC25B', 'CDC42', 'CSF2', 'DPF3', 'EEF1A2', 'EEF1G', 'EIF5A', 'FABP5', 'FGR', 'GSTM4', 'GSTP1', 'HAP1', 'HCK', 'HP', 'HSPB1', 'IGFBP3', 'KIF11', 'KRT17', 'KRT18', 'KRT19', 'KRT8', 'LGALS1', 'MAP4', 'MCM3', 'MDM2', 'MMP7', 'NFKB1', 'P4HB', 'PGK1', 'PRDX1', 'PSMB6', 'RABGGTB', 'RAC1', 'RHOA', 'S100A2', 'TUBB', 'YWHAZ', 'ZBTB17']
Refined Community 17: ['ABCC5', 'ABCF2', 'ADAR', 'AGO2', 'AGPAT1', 'AIP', 'AKT1', 'ALKBH4', 'ANAPC15', 'AP2M1', 'AP4S1', 'APIP', 'ARHGAP5', 'ARHGAP9', 'ARHGEF11', 'ARNT', 'ATF6', 'ATP6V1C1', 'B4GALT3', 'BAZ1A', 'BCL11B', 'BLCAP', 'C14orf132', 'C5orf22', 'CALM1', 'CAPZA2', 'CAT', 'CAV2', 'CCDC107', 'CCDC127', 'CCDC85C', 'CCS', 'CCT5', 'CCT6A', 'CD44', 'CD84', 'CDK4', 'CDK5', 'CERS2', 'CFL2', 'CKS1B', 'CLCN2', 'CLPTM1L', 'COCH', 'COPS6', 'CORO1B', 'CPSF1', 'CRKL', 'CTSF', 'CTSK', 'CUL2', 'DAP', 'DDIT3', 'DDX10', 'DEDD', 'DENND4B', 'DHCR7', 'DPP3', 'DUSP12', 'DVL3', 'DYNLT2B', 'EAPP', 'EEF1AKMT4', 'EGFR', 'EIF4G1', 'FADD', 'FAM131A', 'FAM177A1', 'FASTK', 'FASTKD3', 'FIGNL1', 'FJX1', 'FKBP9', 'GCDH', 'GEMIN2', 'GFUS', 'GNB2', 'GNS', 'GPAA1', 'HAX1', 'HDGF', 'HYOU1', 'IGSF8', 'ILF2', 'INPPL1', 'JPH1', 'JTB', 'JTB-DT', 'KCNH2', 'KDM2A', 'KLC1', 'LAMTOR1', 'LANCL2', 'LPCAT1', 'LSM8', 'LY6E', 'MAF1', 'MAFB', 'MAGEF1', 'MANBAL', 'MBD6', 'MBIP', 'MCM7', 'MED15', 'MET', 'METTL1', 'MFN1', 'MIPOL1', 'MRM2', 'MRPL11', 'MRPL23', 'MRPL36', 'MRPL9', 'MRPS17', 'MTRR', 'MYC', 'MYNN', 'NAT10', 'NAXE', 'NDUFB5', 'NDUFS2', 'NDUFS6', 'NFKBIA', 'NIPSNAP2', 'NKX2-1', 'NPR1', 'NRDE2', 'NUBP1', 'NUDT1', 'NUMA1', 'OPA1', 'OTULIN', 'OTULINL', 'P2RY6', 'PARD3', 'PAX9', 'PBX1', 'PC', 'PCID2', 'PCYT1A', 'PDCD6', 'PHC3', 'PI4KA', 'PI4KB', 'PIK3CA', 'PIP5K1A', 'PLD1', 'PMF1', 'PMVK', 'PNN', 'POGZ', 'POLD4', 'POLR2H', 'POLR2J', 'PPFIA1', 'PPP1CA', 'PPP1R16A', 'PPP1R3D', 'PPP2R3C', 'PRCC', 'PRORP', 'PSMB4', 'PUF60', 'PYGO2', 'RAD23A', 'RBM4', 'RIT1', 'RMI2', 'RPS27', 'RTN4R', 'S100A11', 'S100A13', 'S100A6', 'SDHA', 'SEC23A', 'SEC61G', 'SEC62', 'SETDB1', 'SHC1', 'SIVA1', 'SKIL', 'SLC12A7', 'SLC39A1', 'SLC4A2', 'SMIM12', 'SNAP29', 'SNAPIN', 'SNX6', 'SRD5A1', 'SRP54', 'SSR2', 'STRN3', 'TAF6', 'TARS1', 'TDRKH', 'TECPR2', 'TENT4A', 'TES', 'TFDP1', 'TFRC', 'TMEM134', 'TMEM191A', 'TPM3', 'TRIM44', 'TSFM', 'TTC8', 'TUFT1', 'USP13', 'USP21', 'VEGFA', 'YKT6', 'YY1', 'ZNF277', 'ZNF3', 'ZNF74', 'ZSCAN21']
Refined Community 18: ['ATP10B', 'AURKA', 'COL11A1', 'MCM6', 'SCG5', 'TFPI2', 'TM4SF4', 'TOP2A', 'XAGE1A']
Refined Community 19: ['CLDN5', 'DNAAF1', 'DNAI2', 'DRC3', 'FABP6', 'HIGD1B', 'IQCG', 'MUC4', 'RARRES2', 'SCGB1A1', 'SFTPA2', 'SFTPB', 'SFTPC']
Refined Community 20: ['ABCA12', 'ABCC3', 'ACRV1', 'ADAMDEC1', 'ADCY9', 'ADGRD2', 'ADGRG6', 'ADORA2A', 'ADORA3', 'ALAS2', 'AMN', 'ANKRD36B', 'AOX1', 'APOA2', 'APOBEC3F', 'AQP6', 'ARHGAP17', 'ARHGAP45', 'ARTN', 'ASCL3', 'ATF5', 'ATP2A3', 'ATP6V0A1', 'BASP1', 'BBIP1', 'BBOX1', 'BCL2', 'BRME1', 'BTG2', 'C1QL1', 'C1RL', 'C21orf91', 'CACNA1A', 'CACNA1F', 'CAMKMT', 'CCND2', 'CD180', 'CD19', 'CD300A', 'CD38', 'CD79A', 'CDC42EP2', 'CENPT', 'CEP135', 'CFB', 'CHST11', 'CHST2', 'CMA1', 'CNR1', 'COL9A3', 'COMMD3', 'CPLX3', 'CRADD', 'CRISP1', 'CRYBB3', 'CSH1', 'CSNK2A1', 'CYBA', 'CYP2E1', 'CYTIP', 'DAZ1', 'DCAF1', 'DCAKD', 'DENND5B', 'DERL1', 'DIP2C', 'DIPK1A', 'DKFZP434A062', 'DLG4', 'DLL3', 'DMXL2', 'DNAJB9', 'DNAJC1', 'DPYSL4', 'DRD4', 'DTNB-AS1', 'DTX3', 'EAF2', 'EFEMP2', 'ELF5', 'ENDOU', 'EPS15L1', 'ERP44', 'FAM30A', 'FCRL2', 'FER1L4', 'FFAR2', 'FGF12', 'FKBP11', 'FKBP1B', 'FRMD1', 'GABRR1', 'GAS8-AS1', 'GAST', 'GATA2', 'GCKR', 'GH1', 'GJA3', 'GJD2', 'GPER1', 'GPR31', 'GRWD1', 'GZMB', 'HERPUD1', 'HES2', 'HGH1', 'HPD', 'HSF2BP', 'HSPA13', 'ICAM2', 'ICAM3', 'IDH3A', 'IFNA10', 'IFNAR2', 'IFT70A', 'IGHA1', 'IGHD', 'IGHG1', 'IGHM', 'IGHV1-69', 'IGHV3-20', 'IGHV3-21', 'IGHV3-23', 'IGHV3-33', 'IGHV3-47', 'IGHV3-7', 'IGHV3-72', 'IGHV3-73', 'IGHV4-34', 'IGHV4-61', 'IGKC', 'IGKV1D-13', 'IGKV1D-17', 'IGKV1D-37', 'IGKV1D-39', 'IGKV1OR1-1', 'IGKV1OR2-108', 'IGKV2D-28', 'IGKV3-20', 'IGKV4-1', 'IGLC2', 'IGLL3P', 'IGLV3-19', 'IL5RA', 'IRAK3', 'IRAK4', 'IRF4', 'ISG20', 'ISOC2', 'ITFG2', 'ITGB4', 'ITIH3', 'ITM2C', 'JCHAIN', 'KAT6B', 'KCNJ14', 'KCNJ4', 'KCNN3', 'KIR3DL2', 'KIZ', 'KLK6', 'KRT81', 'L1CAM', 'LAX1', 'LDOC1', 'LIME1', 'LRRC36', 'LRRC41', 'LZTS1', 'MAGOH2P', 'MANF', 'MAP2K7', 'MAST1', 'MAVS', 'MBD2', 'MED1', 'MED13L', 'MGAT2', 'MMP14', 'MRPS31', 'MSX2', 'MYL10', 'N4BP1', 'NANS', 'NCK1', 'NCR1', 'NCR3', 'NEBL', 'NEU2', 'NF1', 'NLE1', 'NMNAT2', 'NOS1', 'NPEPL1', 'NPPC', 'NR1I2', 'NSG1', 'NXPH3', 'NXPH4', 'OGDHL', 'OGFRL1', 'ORAI2', 'OXCT1', 'PAK3', 'PAOX', 'PAX5', 'PCYT1B', 'PDK1', 'PGLYRP1', 'PIM2', 'PLA2G2A', 'PLXDC1', 'PNOC', 'POU2AF1', 'PPBPP2', 'PPT2', 'PPY', 'PRDM1', 'PRDX4', 'PRMT2', 'PROZ', 'PTGDR2', 'PTPN2', 'RAB26', 'RABAC1', 'RFPL3', 'RHAG', 'RHOH', 'RHOQ', 'RMDN1', 'RRP9', 'RYBP', 'SAA1', 'SAA3P', 'SCT', 'SEC14L1', 'SEC14L4', 'SEC24A', 'SEL1L', 'SEL1L3', 'SERPINB3', 'SERPINB4', 'SH3BP2', 'SH3D21', 'SIGLEC5', 'SKAP1', 'SLAMF7', 'SLC12A3', 'SLC1A1', 'SLC1A4', 'SLC22A2', 'SLC26A4', 'SLC2A5', 'SLC35E3', 'SLC38A7', 'SLC43A1', 'SLC6A13', 'SMIM27', 'SNN', 'SOD2', 'SOX10', 'SOX14', 'SPAG4', 'SPANXA1', 'SPATS2L', 'SPCS3', 'SPPL2B', 'SRF', 'SRRM1', 'SSR3', 'SSR4', 'SSX2', 'ST6GAL1', 'STXBP6', 'TAPBPL', 'TCL1A', 'TCL1B', 'TCP10L', 'TCP11L1', 'TENT5C', 'TFAP2A', 'THEMIS2', 'THRA', 'TMCO3', 'TNFRSF17', 'TNFRSF6B', 'TNIP3', 'TP63', 'TPP1', 'TPSG1', 'TRAM2', 'TRAPPC9', 'TRBV16', 'TRGC1', 'TRGV5P', 'TRIM26', 'TRIM31', 'TTLL4', 'TXNDC15', 'TXNDC5', 'UBE2J1', 'UMOD', 'VCPIP1', 'VPREB1', 'VPS13D', 'WBP11', 'WBP4', 'YWHAH-AS1', 'ZBP1', 'ZBTB25', 'ZC3H13', 'ZER1', 'ZFYVE9', 'ZNF215', 'ZNF419', 'ZNF473', 'ZSWIM8-AS1']
Refined Community 21: ['AASS', 'ABCA3', 'ABCA4', 'ABCC6', 'ADCY7', 'ADGRF5', 'AHCYL2', 'AK1', 'ALDH3B1', 'ALDH5A1', 'ALDH8A1', 'ALOX15B', 'APLP2', 'APOD', 'AQP3', 'AQP4', 'ARHGEF10', 'ARMC9', 'ARRB1', 'ATP11A', 'ATP2B4', 'ATP8A1', 'ATXN2', 'ATXN7', 'BLVRA', 'BMP3', 'BTG3', 'C1orf116', 'CACNB1', 'CAMTA2', 'CAPN2', 'CAVIN2', 'CCNJL', 'CD55', 'CDC42BPA', 'CDKL2', 'CDS1', 'CEACAM8', 'CEBPD', 'CEP112', 'CFAP410', 'CHAT', 'CHTOP', 'CISH', 'CITED2', 'CLOCK', 'CPM', 'CYB5A', 'CYP2B6', 'CYP2B7P', 'DAAM2', 'DLC1', 'DLEC1', 'DPY19L1', 'DUSP1', 'EDEM1', 'EFEMP1', 'ELK3', 'EN2', 'EPB41L1', 'EPHX1', 'EPHX3', 'ERBB2', 'ESR1', 'EVPL', 'FAM13A', 'FAM184A', 'FBP1', 'FGF4', 'FHOD1', 'FMO5', 'FOLR1', 'FTH1', 'FYCO1', 'GALNT11', 'GEM', 'GGCX', 'GLS', 'GNA14', 'GON4L', 'GPD1L', 'GPR39', 'GPRC5A', 'HAUS4', 'HDAC5', 'HOPX', 'HSD17B11', 'HSPB8', 'ICAM1', 'ICAM4', 'ICAM5', 'IL27RA', 'IL6R', 'INPPL1', 'ITPKA', 'KCNS3', 'KLHDC10', 'KLHL20', 'LARP4B', 'LIMCH1', 'LMO3', 'LPCAT1', 'LPIN2', 'LRRC20', 'LTA4H', 'LTK', 'LUZP4', 'MALL', 'MAOA', 'MAP1LC3C', 'MAP3K6', 'MBIP', 'ME3', 'MEGF9', 'MEIS3P1', 'MR1', 'NAAA', 'NAB2', 'NAV2', 'NDNF', 'NEDD4L', 'NEDD9', 'NFKBIA', 'NINJ2', 'NKX2-1', 'NOS3', 'NOTCH2NLA', 'NPR1', 'NR3C1', 'PARM1', 'PBXIP1', 'PELI1', 'PER1', 'PIAS3', 'PIGA', 'PLS3', 'PMM1', 'PON1', 'PON2', 'PON3', 'PPP2R3A', 'PRKCD', 'PSG3', 'PSG4', 'QKI', 'RAB27A', 'RAB38', 'RANBP17', 'RAPGEF2', 'RASA4', 'RBPMS', 'REL', 'RHOBTB2', 'RIMS3', 'RO60', 'RPS6KA2', 'RUFY3', 'S100A13', 'SCEL', 'SCRN1', 'SEC24D', 'SFTPA2', 'SFTPB', 'SFTPD', 'SGSM2', 'SLC16A4', 'SLC25A16', 'SLC34A2', 'SLC39A8', 'SLC7A10', 'SMARCD3', 'SPIDR', 'SPINK2', 'SPTB', 'ST3GAL5', 'STBD1', 'STEAP4', 'SYNE1', 'TIPARP', 'TLE2', 'TMEM243', 'TNS2', 'TRIM2', 'TTC19', 'UBAP1', 'UBAP2L', 'UNC13B', 'UNC93B1', 'VAMP5', 'WNK1', 'WSB1', 'XAGE1B', 'ZBTB16', 'ZER1', 'ZFAND5', 'ZNF254', 'ZNF451', 'ZNF750']
Refined Community 22: ['ADGRL1', 'ADGRL3', 'ANKS1A', 'AP1M2', 'ARHGEF18', 'ARID3A', 'ARID4A', 'ARMCX5', 'ASIC3', 'CA10', 'CD2BP2', 'CDIP1', 'CDK16', 'CERS4', 'CHERP', 'DCAF15', 'DHRS2', 'DLX4', 'DNAJB1', 'DNAJB2', 'EFS', 'EPOR', 'F11R', 'FBRS', 'FZR1', 'GGA2', 'GNA11', 'HAUS5', 'HOOK2', 'KDM4B', 'KLHL26', 'L3MBTL1', 'LCMT1', 'MKNK2', 'NPIPB3', 'NR2F6', 'OTUD3', 'PDK2', 'PGAP1', 'PKP4', 'PLPPR2', 'RALGAPA1', 'RANBP3', 'RIPK4', 'RTN2', 'RUFY3', 'SEZ6L2', 'SHC2', 'SIN3B', 'SLC25A23', 'SLC29A1', 'SMARCA4', 'SMIM7', 'SYT17', 'TLE2', 'TSN', 'VSIG10', 'WDR33', 'XAB2', 'YIPF2', 'ZFP37', 'ZNF177', 'ZNF34', 'ZNF358', 'ZNF443', 'ZNF444', 'ZNF518A', 'ZNF562', 'ZSCAN18']
Refined Community 23: ['ACD', 'ACOT7', 'ACP1', 'ACTR3', 'ADM', 'ADRM1', 'AFP', 'AGMAT', 'AGPAT5', 'AK4', 'ANGPT2', 'ANKLE2', 'AP2B1', 'AP2S1', 'APOBEC3B', 'ARFGEF2', 'ARL4D', 'ARPP19', 'ARTN', 'ASPM', 'ATAD2', 'ATG5', 'ATP2B1', 'AURKA', 'AURKB', 'BIRC5', 'BPNT2', 'BRCA2', 'BRIP1', 'BUB1', 'BUB1B', 'BUB3', 'C6orf120', 'CAD', 'CALM3', 'CARHSP1', 'CBS', 'CBX5', 'CCDC59', 'CCDC93', 'CCL7', 'CCNA2', 'CCNB1', 'CCNB2', 'CCNE1', 'CCNE2', 'CCT5', 'CD70', 'CDC20', 'CDC25A', 'CDC25C', 'CDC27', 'CDC45', 'CDC6', 'CDC7', 'CDCA3', 'CDCA8', 'CDK1', 'CDK5R1', 'CDKN2A', 'CDKN2C', 'CDKN2D', 'CDKN3', 'CDT1', 'CDYL', 'CENPA', 'CENPE', 'CENPF', 'CENPI', 'CENPM', 'CENPN', 'CENPQ', 'CENPU', 'CEP43', 'CEP55', 'CHEK1', 'CHRNA5', 'CIAO1', 'CKAP2', 'CKS2', 'CLPB', 'CMAS', 'CMC2', 'CNOT9', 'COL2A1', 'COMMD8', 'CORT', 'CPT1A', 'CSE1L', 'CSNK1D', 'CTSV', 'DARS1', 'DBF4', 'DCK', 'DDA1', 'DENND1A', 'DEPDC1', 'DGUOK', 'DLGAP5', 'DNA2', 'DNAJA1', 'DNAJC9', 'DNM1L', 'DR1', 'DSN1', 'DSP', 'DTL', 'DTYMK', 'DUSP9', 'E2F8', 'EGLN3', 'EIF3JP1', 'EIF4A3', 'EIF4E2', 'EIF4EBP1', 'EIF5B', 'EIPR1', 'ELOVL4', 'ELOVL6', 'ENPP1', 'ERCC6L', 'ERN1', 'ERO1A', 'ESM1', 'EXO1', 'EXOSC2', 'EZH2', 'F12', 'FAM216A', 'FANCA', 'FANCI', 'FBXO11', 'FBXO5', 'FCHO1', 'FIP1L1', 'FKBP4', 'FOXG1', 'FOXM1', 'FUT9', 'GABPB1', 'GAPDH', 'GCFC2', 'GCH1', 'GGH', 'GINS1', 'GINS2', 'GINS3', 'GINS4', 'GLMN', 'GMFB', 'GNAI3', 'GNAS', 'GOLT1B', 'GPD2', 'GPR19', 'GPR37', 'GPSM2', 'GSS', 'GTSE1', 'H2AC11', 'H2AX', 'H2AZ1', 'H2BC9', 'HAT1', 'HAUS3', 'HBS1L', 'HCCS', 'HDAC2', 'HDGFL3', 'HELLS', 'HEXIM1', 'HJURP', 'HK2', 'HMGA1', 'HMGB2', 'HMMR', 'IGF2BP3', 'IL18RAP', 'IL1RAP', 'IL2RA', 'INTS13', 'IPO5', 'ITCH', 'JPT1', 'KAZALD1', 'KCTD5', 'KIF11', 'KIF14', 'KIF15', 'KIF18A', 'KIF18B', 'KIF20A', 'KIF23', 'KIF2A', 'KIF2C', 'KIF4A', 'KLC1', 'KLRC1', 'KMT5AP1', 'KNTC1', 'KPNA2', 'KRR1', 'KRT8P11', 'KRT8P17', 'LDHB', 'LDHC', 'LMNB1', 'LRFN4', 'LRIF1', 'LRPPRC', 'LSM4', 'MAD2L1', 'MAGEA3', 'MAGOHB', 'MAP6D1', 'MAPKAPK5', 'MCM10', 'MCM4', 'MCM5', 'MCM6', 'MED8', 'MELK', 'MEMO1', 'MFSD12', 'MIF', 'MKI67', 'MMP12', 'MOB1A', 'MPHOSPH9', 'MRGBP', 'MRPL12', 'MRPL13', 'MRPL19', 'MRPL42', 'MSH6', 'MTCH2', 'MTF2', 'MTHFD2', 'MTIF2', 'MTPAP', 'MUC16', 'MYBL1', 'MYBL2', 'MYO7A', 'NAA50', 'NAB1', 'NABP2', 'NBN', 'NCAPD2', 'NCAPD3', 'NCAPG', 'NCAPG2', 'NCAPH', 'NCBP1', 'NDC80', 'NDUFA9', 'NEIL3', 'NEK2', 'NEMP1', 'NETO2', 'NME1', 'NMU', 'NOLC1', 'NRAS', 'NSD2', 'NTAQ1', 'NUDT3', 'NUP210', 'NUP37', 'NUP62', 'NUSAP1', 'OIP5', 'OR7E12P', 'ORC1', 'PA2G4', 'PAICS', 'PARPBP', 'PAWR', 'PAX8', 'PBK', 'PCLAF', 'PCMT1', 'PCNA', 'PDCD2', 'PFKP', 'PFN2', 'PGK1', 'PGM3', 'PHF10', 'PHTF2', 'PIMREG', 'PITX1', 'PKM', 'PKP2', 'PLIN2', 'PLK4', 'PLOD2', 'PMAIP1', 'PNKP', 'PNP', 'POLE2', 'POLR3G', 'POMT2', 'PPARD', 'PPAT', 'PPFIA1', 'PPIAP21', 'PPID', 'PPIF', 'PPM1G', 'PPP2R3C', 'PRC1', 'PRIM1', 'PRIM2', 'PRKAA2', 'PRKDC', 'PRMT1', 'PRPS1', 'PRR7', 'PSAT1', 'PSMA5', 'PSMA7', 'PSMB2', 'PSMD12', 'PSME3', 'PSRC1', 'PTBP3', 'PTTG1', 'PTTG2', 'PTTG3P', 'PVR', 'PYGL', 'QSER1', 'R3HDM1', 'RAB22A', 'RACGAP1', 'RAD51AP1', 'RAD54L', 'RALA', 'RAN', 'RASAL2', 'RBL1', 'RC3H2', 'RECQL', 'RECQL4', 'RFC2', 'RFC3', 'RFC4', 'RGS20', 'RIF1', 'RIPK2', 'RNMT', 'RPP25', 'RRM2', 'RTCA', 'RUVBL2', 'SAP30', 'SAR1A', 'SCD', 'SEC23A', 'SEMA3A', 'SEPHS1', 'SFXN1', 'SHCBP1', 'SKA1', 'SLBP', 'SLC16A1', 'SLC16A3', 'SLC25A10', 'SLC2A1', 'SLC38A1', 'SLC7A5', 'SLC9A2', 'SMC2', 'SMC6', 'SMG1P5', 'SMNDC1', 'SNCG', 'SNRPA1', 'SNRPB', 'SNX6', 'SNX7', 'SOD2', 'SPAG5', 'SPC25', 'SRD5A1', 'SRM', 'SS18', 'SSX2IP', 'STC1', 'STC2', 'STEAP1B', 'STMN1', 'STRN4', 'SYNCRIP', 'SZRD1', 'TACC3', 'TAF7L', 'TBC1D31', 'TBRG4', 'TCP1', 'TEAD4', 'TEX30', 'TFDP1', 'TFG', 'TFPT', 'THOP1', 'TIMM8A', 'TK1', 'TMEFF1', 'TMEM38B', 'TNNT1', 'TOMM40', 'TOP2A', 'TPRKB', 'TPX2', 'TRIP13', 'TRMU', 'TROAP', 'TSR1', 'TTF2', 'TTK', 'TUBA3C', 'TUBA3E', 'TUBA4A', 'TUBA4B', 'TUBG1', 'TWF1', 'TWSG1', 'TYMS', 'UBA6', 'UBAC1', 'UBE2C', 'UBE2D1', 'UBE2K', 'UBE2N', 'UBE2S', 'UBE2V1', 'UBE2V2', 'UGT8', 'VANGL1', 'VEGFA', 'VRK2', 'WAPL', 'WARS1', 'WASF1', 'WDHD1', 'WDR43', 'WDR62', 'WDR76', 'XPNPEP1', 'XRCC3', 'YEATS2', 'YEATS4', 'YKT6', 'YME1L1', 'ZNF248', 'ZSCAN5A', 'ZWILCH', 'ZWINT']
Refined Community 24: ['ALMS1', 'ATP8B1', 'CCDC81', 'DCLRE1C', 'DUOX1', 'GTF2H3', 'LMO4', 'PRR11', 'SPN', 'TCAF1', 'TSR1', 'ZC3H7B', 'ZNF160', 'ZNF611']
Refined Community 25: ['ACTR2', 'AIDA', 'ANXA3', 'ARL6IP5', 'ATP6AP2', 'C6', 'CCT2', 'CD164', 'CD55', 'CPNE3', 'CXCL2', 'CXCL8', 'DEFB1', 'DMD', 'DNAJC12', 'DUSP6', 'FCGR3A', 'FGF14', 'FOS', 'FXR1', 'NELL2', 'PLA2G4A', 'PPBP', 'RAB1A', 'RGS1', 'SERPINA1', 'SLC39A8', 'SLC4A4', 'SOX9', 'TMED2', 'TMEM47', 'TOB1', 'TPD52', 'TRAM1', 'TWF1', 'UBE2J1', 'UBXN4', 'UGCG', 'YWHAE']
Refined Community 26: ['ACP5', 'ADAM15', 'AGAP3', 'AGO2', 'AGPAT1', 'APOC1', 'ATG7', 'AXIN2', 'BASP1', 'BCL2L11', 'CAVIN1', 'CCL23', 'CCL3', 'CCN3', 'CD68', 'CFL1', 'CLEC4A', 'COTL1', 'CSF2RB', 'CSTB', 'CTSD', 'CTSK', 'CTSS', 'CTSZ', 'F7', 'GJA5', 'IFITM10', 'IQGAP1', 'ITGAX', 'LCN2', 'LILRA4', 'LPL', 'LRG1', 'LRP5', 'LY75', 'LYZ', 'MPEG1', 'MYO7A', 'PDIA4', 'PIGT', 'PLD3', 'PLET1', 'POU3F2', 'PSMA4', 'PTGDR2', 'RAP2B', 'RASGRP2', 'SAT1', 'SCD', 'SEMA6A', 'SH3RF1', 'SMAP2', 'SNCB', 'SND1', 'SOCS3', 'SP4', 'SPP1', 'SYK', 'TBL3', 'TCEAL9', 'TLE5', 'USF1', 'WFDC21P']
Refined Community 27: ['ACTN4', 'AGPAT5', 'ANK3', 'ANTXR2', 'APBB2', 'APLP2', 'ARHGAP21', 'ARPC3', 'ATXN2', 'C1QTNF3', 'C22orf39', 'C5orf34', 'C6orf120', 'CADM1', 'CAST', 'CCNL2', 'CD200', 'CGGBP1', 'CHD8', 'CHI3L1', 'CLK1', 'CLOCK', 'CTCF', 'DDC', 'DDX3X', 'DNAJC10', 'DPM1', 'EIF1AX', 'EIF3M', 'ELOC', 'EPHB6', 'ETNK1', 'EXT1', 'FXN', 'GFRA2', 'GINM1', 'GLG1', 'GTF2E2', 'H2AZ2', 'HNRNPC', 'HNRNPH1', 'HTRA1', 'IFNAR2', 'IFT20', 'ING1', 'IREB2', 'ITIH4', 'JKAMP', 'KAT2B', 'KDR', 'KIF5B', 'KLF13', 'KMT2A', 'LATS2', 'LIMS1', 'MAPK1', 'MDM4', 'MED23', 'MEF2D', 'MINDY1', 'MOCS2', 'MPV17', 'MTCH1', 'MTHFR', 'NFIB', 'NME3', 'NRAS', 'NRIP1', 'OGN', 'OSBPL11', 'OTUD5', 'PAIP2', 'PARP6', 'PDGFA', 'PGRMC1', 'PIAS1', 'PIAS2', 'PJA1', 'PKD2', 'PLD1', 'PNN', 'PRMT1', 'PRPSAP2', 'PTGS1', 'PTPRD', 'PTPRS', 'RASA1', 'RCN1', 'RELL1', 'RNF149', 'RNF6', 'RRP1B', 'SCARB2', 'SERPINA10', 'SF3B1', 'SIRPA', 'SLC22A23', 'SLC35A1', 'SLC4A5', 'SMAD4', 'SMO', 'SPPL3', 'SRSF1', 'STX3', 'STXBP3', 'THRA', 'TIA1', 'TM7SF3', 'TMCO1', 'TNPO2', 'TPRG1L', 'TPST1', 'TUG1', 'UBE2V2', 'UPF3B', 'VPS26B', 'VPS4A', 'WDR26', 'WDR48', 'WDR81', 'WWC2', 'YIPF5', 'YTHDC1', 'ZNF131', 'ZNF136', 'ZNF639', 'ZSCAN26']
Refined Community 28: ['ABCA1', 'ABCC1', 'ABLIM1', 'ACE', 'ACKR2', 'ACKR3', 'ACKR4', 'ACSL1', 'ACTA1', 'ACTA2', 'ACTC1', 'ACVR2A', 'ACVRL1', 'ADARB1', 'ADCY8', 'ADGRE5', 'ADH1A', 'ADIPOQ', 'ADRB2', 'ADRB3', 'AHNAK', 'AHR', 'AKAP12', 'ALAS2', 'ALDH1A1', 'ALDOB', 'ANGPT1', 'ANGPTL2', 'ANKRD1', 'ANKRD10', 'ANKRD33B', 'ANKRD40', 'ANTXR2', 'AQP1', 'ARHGEF3', 'ARMCX2', 'ARRB1', 'ART3', 'ATOH7', 'ATP1A2', 'ATP2A2', 'BCL6B', 'BDNF', 'BMP6', 'BNC1', 'BPGM', 'BPIFB1', 'BUB1', 'C3', 'CA2', 'CA3', 'CACNB2', 'CADM1', 'CALCR', 'CALCRL', 'CAV1', 'CAV3', 'CAVIN1', 'CAVIN2', 'CAVIN3', 'CCKAR', 'CCL21', 'CCN1', 'CCN5', 'CCRL2', 'CD47', 'CD93', 'CDH11', 'CDH5', 'CDK14', 'CDKN1C', 'CDO1', 'CDR2', 'CES1', 'CES2', 'CFD', 'CFH', 'CFHR1', 'CIDEC', 'CKMT2', 'CLDN5', 'CLEC3B', 'CLIC4', 'CNTN1', 'COL13A1', 'COL1A1', 'COL1A2', 'COL3A1', 'COL6A2', 'COX7A1', 'CP', 'CPE', 'CRHR1', 'CRIP1', 'CXCL14', 'CXCR4', 'CYP2A6', 'CYP2B6', 'CYP2E1', 'CYP2F1', 'CYP2S1', 'CYP4B1', 'CYTH3', 'DCN', 'DENND2B', 'DENND4C', 'DIPK2A', 'DNM1', 'DPEP1', 'DPT', 'DUSP1', 'EDN1', 'EDNRB', 'EFNB2', 'EMCN', 'EMP2', 'ENAH', 'ENG', 'ENPP2', 'EPAS1', 'EPB41', 'EPHA5', 'EPS15', 'ETS1', 'ETS2', 'FAS', 'FERMT2', 'FEZ2', 'FGF7', 'FGF9', 'FGL2', 'FHL1', 'FKBP9', 'FLT1', 'FMO1', 'FMO3', 'FOXF1', 'FOXF2', 'FXYD1', 'FYN', 'G0S2', 'GADD45B', 'GBP2', 'GBP4', 'GFRA2', 'GGH', 'GIMAP4', 'GLUL', 'GMFG', 'GNAS-AS1', 'GNB4', 'GNG11', 'GNG2', 'GPAM', 'GPC3', 'GPM6B', 'GPR182', 'GPX3', 'GREM2', 'GRK5', 'GSN', 'GSTM1', 'GUCY1B1', 'GYG1', 'H2AX', 'H2BC4', 'HBB', 'HCK', 'HEPH', 'HEY1', 'HNRNPA1L2', 'HOPX', 'HOXA5', 'HOXA6', 'HOXB5', 'HP', 'HSD11B1', 'ICAM2', 'ID3', 'IFI16', 'IFIH1', 'IFIT3', 'IFITM3', 'IGFBP2', 'IGFBP5', 'IGFBP6', 'IGHD', 'IL11RA', 'IL1B', 'IL27RA', 'IL6ST', 'INMT', 'INPP5A', 'ITPKB', 'JUN', 'KALRN', 'KANK3', 'KCTD12', 'KDR', 'KIT', 'KITLG', 'KLF2', 'KLF4', 'KLF7', 'KLF9', 'KRT13', 'KRT4', 'KRT85', 'LAMA2', 'LAMB1', 'LATS2', 'LEPR', 'LIFR', 'LIMCH1', 'LIN9', 'LMO2', 'LORICRIN', 'LOX', 'LOXL1', 'LTB', 'LTBP4', 'LYL1', 'LYSMD2', 'LYVE1', 'MACF1', 'MAP4', 'MAP7D1', 'MAPT', 'MARCKS', 'MEF2C', 'MEIS1', 'MEOX2', 'METAP1', 'MFAP2', 'MFAP5', 'MFHAS1', 'MGP', 'MPDZ', 'MS4A1', 'MS4A6A', 'MSLN', 'MTSS2', 'MYB', 'MYH1', 'MYH11', 'MYH6', 'MYL3', 'MYL4', 'MYL7', 'MYL9', 'MYO1B', 'MYO6', 'MYZAP', 'NDN', 'NDRG2', 'NDST1', 'NFIB', 'NFKBIA', 'NID1', 'NOTCH4', 'NPNT', 'NPR3', 'NR2F2', 'NT5DC2', 'NTN1', 'NUMB', 'OGN', 'OMD', 'PAG1', 'PAM', 'PAPSS2', 'PCDHA9', 'PCK1', 'PDGFRA', 'PDGFRB', 'PDLIM1', 'PEG3', 'PGM2', 'PGRMC1', 'PKD2', 'PKIA', 'PLAC9', 'PLPP1', 'PLPP3', 'PLTP', 'PMP22', 'PON1', 'POSTN', 'PPP1CB', 'PPP2R3C', 'PRDX6', 'PRKCE', 'PROM1', 'PRX', 'PSIP1', 'PSMB10', 'PTCH1', 'PTGES', 'PTGIS', 'PTPRB', 'PTPRD', 'QKI', 'RAB12', 'RAB28', 'RABGGTA', 'RAMP2', 'RARRES2', 'RASIP1', 'RBP1', 'RCN1', 'RDH11', 'RECK', 'REG3G', 'RGS2', 'RHOB', 'RHOJ', 'RIPOR2', 'ROBO1', 'RPTN', 'S100A8', 'S100A9', 'SASH1', 'SATB1', 'SC5D', 'SCEL', 'SCGB1A1', 'SCN7A', 'SEMA3C', 'SEMA7A', 'SEPTIN4', 'SERPINA3', 'SERPING1', 'SESN1', 'SH3BP5', 'SHE', 'SHOX2', 'SIAH1', 'SLC10A2', 'SLC4A5', 'SLC7A5', 'SLCO3A1', 'SMAD6', 'SMARCA2', 'SNCA', 'SOD3', 'SORBS1', 'SOX11', 'SOX17', 'SOX2', 'SPA17', 'SPARC', 'SPARCL1', 'SPEF1', 'SPIB', 'SPOCK2', 'SPTAN1', 'SPTBN1', 'SRGN', 'SSPN', 'ST8SIA4', 'STAB1', 'STMN2', 'STMN3', 'SULT1A1', 'SULT1D1P', 'SURF2', 'TAGLN', 'TANGO2', 'TBX2', 'TBX3', 'TCF21', 'TCF3', 'TCF4', 'TEK', 'TEKT1', 'TENT5C', 'TFPI', 'TFRC', 'THBD', 'TIAM1', 'TIE1', 'TIMP3', 'TJP1', 'TM2D3', 'TMEFF1', 'TMEM45A', 'TMEM71', 'TNFRSF19', 'TNNC1', 'TNNI3', 'TNNT2', 'TNNT3', 'TNS2', 'TNXB', 'TOP2A', 'TPH1', 'TPRG1L', 'TRBC1', 'TSPAN13', 'TSPAN6', 'TSPAN7', 'TUBA1A', 'TWSG1', 'UPK3B', 'USF2', 'USP18', 'VAMP3', 'VAX1', 'VCL', 'VEGFA', 'VEGFD', 'VWF', 'WWTR1', 'XIST', 'ZBTB16', 'ZBTB20', 'ZBTB46', 'ZEB1', 'ZMYND11']
Refined Community 29: ['AASS', 'ACADL', 'ACE2', 'ACLY', 'ACSL4', 'ACSL5', 'ACTN1', 'ACTN4', 'ADAM19', 'ADCY7', 'ADGRG1', 'ADIPOR2', 'ADSS1', 'AK1', 'ALDOA', 'ALDOC', 'ANK3', 'ANXA4', 'APEX1', 'APOC1', 'AREG', 'ARG1', 'ARG2', 'ARGLU1', 'ARL8B', 'ATOX1', 'ATP11A', 'ATP1A1', 'ATP5F1C', 'ATP6V0A1', 'ATP6V0C', 'ATP6V0D1', 'ATP6V1C1', 'ATXN10', 'AVPI1', 'AXIN1', 'AXL', 'AZIN1', 'B3GAT3', 'B4GALNT1', 'BASP1', 'BBLN', 'BCL2A1', 'BEX1', 'BEX4', 'BHLHE40', 'BMP4', 'BRD7', 'BSG', 'BST1', 'BTG1', 'BTG3', 'C16orf89', 'C17orf49', 'C1QB', 'C1QC', 'C5', 'CA8', 'CAMSAP1', 'CAPZA3', 'CASK', 'CCDC186', 'CCL15', 'CCL23', 'CCND1', 'CCR5', 'CCT3', 'CD14', 'CD44', 'CD63', 'CD68', 'CD74', 'CD9', 'CDK2AP2', 'CDKN1A', 'CEACAM1', 'CEBPA', 'CES3', 'CH25H', 'CHCHD7', 'CHD4', 'CHI3L1', 'CHIA', 'CHL1', 'CHRNB1', 'CIP2A', 'CITED2', 'CKMT1B', 'CKS2', 'CLCN5', 'CLDN3', 'CLDN7', 'CLDND1', 'CLIC1', 'CLIP4', 'CLU', 'CNDP2', 'CNIH2', 'COL15A1', 'COL18A1', 'COTL1', 'CPOX', 'CRB3', 'CRLF1', 'CRYGD', 'CSF2', 'CSF2RB', 'CSRP2', 'CSTB', 'CTNND2', 'CTSA', 'CTSB', 'CTSC', 'CTSD', 'CTSH', 'CTSK', 'CTSS', 'CTSZ', 'CYB5R1', 'CYB5R3', 'CYBA', 'CYRIB', 'DAP', 'DLK1', 'DSC2', 'DUSP6', 'EDEM1', 'EEF1D', 'EEF2', 'EHMT2', 'EIF1AX', 'EIF2AK4', 'EIF3E', 'EIF4B', 'EIF4G1', 'ELF5', 'ELL2', 'ELOVL1', 'ENO1', 'ENTPD1', 'EPCAM', 'EPHA7', 'ERH', 'ERRFI1', 'ESRP1', 'ETV2', 'F10', 'F3', 'F7', 'FAM117A', 'FAM162A', 'FAM3C', 'FASN', 'FBP2', 'FCGR2B', 'FKBP2', 'FKBP4', 'FMR1', 'FNTA', 'FPR2', 'FUCA1', 'G6PD', 'GADD45A', 'GALNT3', 'GAPDH', 'GARS1', 'GCH1', 'GFUS', 'GGCX', 'GJA1', 'GJA3', 'GJB2', 'GJB3', 'GLRX', 'GNE', 'GNL3', 'GNS', 'GOLM1', 'GPI', 'GRHPR', 'GRINA', 'GSTT1', 'H19', 'H2AC8', 'HAP1', 'HDC', 'HDLBP', 'HEXA', 'HHEX', 'HIBADH', 'HIF1A', 'HK2', 'HLA-DMA', 'HLA-DMB', 'HLA-DQA2', 'HLA-DQB1', 'HLA-DRA', 'HLA-DRB1', 'HM13', 'HMGB3', 'HMGN1', 'HNF1B', 'HOXD1', 'HPN', 'HSPA1B', 'HSPA5', 'HSPA8', 'HSPA9', 'HSPH1', 'IBSP', 'ID2', 'IFI30', 'IGFBP3', 'IGHA2', 'IGHG1', 'IGHV1-2', 'IGHV1-24', 'IGKV2D-29', 'IGLV1-50', 'IL11', 'IL13RA2', 'IL18', 'IL4R', 'INHBB', 'IQGAP1', 'ITGA4', 'ITGA8', 'ITGAX', 'ITGB2', 'ITIH4', 'ITM2C', 'ITPR2', 'KCNJ15', 'KCNK1', 'KDELR1', 'KLF5', 'KLHDC2', 'KNG1', 'KRAS', 'KRT18', 'KRT7', 'KRT8', 'LAMB3', 'LAMC2', 'LAP3', 'LAPTM5', 'LAS1L', 'LBP', 'LCN2', 'LCP1', 'LDHA', 'LGALS3', 'LITAF', 'LPCAT1', 'LPCAT3', 'LRG1', 'LRP2', 'LRRFIP1', 'LY6D', 'LY75', 'MAN1A1', 'MANF', 'MAPK1', 'MAPRE1', 'MARCHF5', 'MARCO', 'MATN4', 'MBTD1', 'MDFI', 'MDFIC', 'ME1', 'MEF2B', 'MEG3', 'MIA2', 'MIEN1', 'MLEC', 'MMP12', 'MPEG1', 'MRC1', 'MRPS18B', 'MRPS34', 'MSR1', 'MT1F', 'MT1X', 'MTIF2', 'MUC1', 'MYDGF', 'MYH7', 'NABP1', 'NAGK', 'NAPSA', 'NCL', 'NDUFAF4', 'NEK4', 'NFIL3', 'NHSL3', 'NME2', 'NMT1', 'NNT', 'NPC2', 'NPDC1', 'NR2F1', 'NUCB2', 'NUDT4', 'NUP88', 'ORM1', 'OSBPL1A', 'OSTF1', 'PABPC1', 'PAFAH1B3', 'PAPOLA', 'PARM1', 'PCBD1', 'PCYOX1', 'PCYT1A', 'PDIA6', 'PDK3', 'PFDN2', 'PFKL', 'PGK1', 'PGLS', 'PGLYRP1', 'PHB2', 'PHLDA1', 'PHLDA2', 'PIGA', 'PIP4K2C', 'PIP5K1B', 'PISD', 'PKHD1', 'PKM', 'PLA2G5', 'PLA2G7', 'PLBD1', 'PLD3', 'PLET1', 'PLIN2', 'PLP2', 'PLXNB2', 'POLG', 'POLR1C', 'POLR2E', 'PON2', 'PPARG', 'PPP1R14B', 'PPP2R5C', 'PRB3', 'PRCC', 'PRDX4', 'PRDX5', 'PRELID1', 'PRNP', 'PRXL2A', 'PSAP', 'PSAT1', 'PSCA', 'PSEN1', 'PSMB5', 'PSMD4', 'PSMD5', 'PSME1', 'PTGR1', 'PTGS1', 'PTPRF', 'RABGGTB', 'RACK1', 'RAP1GAP', 'RBP4', 'RDH11', 'REEP6', 'RFK', 'RGCC', 'RIDA', 'RNASE3', 'RNASET2', 'RNF149', 'RNF181', 'RNF4', 'RO60', 'ROS1', 'RPL10A', 'RPL17', 'RPL28', 'RPL3', 'RPL36', 'RPL37', 'RPL6', 'RPL8', 'RPS18', 'RPS2', 'RPS8', 'RRBP1', 'S100A1', 'S100G', 'SAT1', 'SCAMP1', 'SCD', 'SCG3', 'SDC1', 'SEC23B', 'SERPINE1', 'SERPINE2', 'SFTPB', 'SHC1', 'SHMT1', 'SIRPA', 'SIVA1', 'SLAIN1', 'SLC12A2', 'SLC15A2', 'SLC16A1', 'SLC25A39', 'SLC31A1', 'SLC34A2', 'SLC38A2', 'SLC4A4', 'SLPI', 'SND1', 'SNX10', 'SOAT1', 'SOCS2', 'SPECC1', 'SPG21', 'SPINT1', 'SPP1', 'SRSF6', 'ST13', 'ST3GAL4', 'ST6GAL1', 'ST7', 'STARD10', 'STXBP2', 'TACSTD2', 'TANK', 'TAOK3', 'TBC1D24', 'TCEAL9', 'TES', 'TFCP2L1', 'TGFBI', 'TGIF1', 'TGOLN2', 'THBS1', 'TLCD4', 'TLE5', 'TM2D2', 'TMEM268', 'TMEM30A', 'TMEM30B', 'TMEM50B', 'TMEM62', 'TNFAIP1', 'TNFSF9', 'TNNT1', 'TOB1', 'TOM1L1', 'TPI1', 'TPM4', 'TRBC1', 'TSPAN8', 'TSR1', 'TULP2', 'TYROBP', 'UBQLN2', 'UBXN1', 'UOX', 'VAMP2', 'VAMP8', 'VASP', 'VEGFB', 'VIL1', 'VMP1', 'WFDC21P', 'WLS', 'XBP1', 'ZDHHC3', 'ZDHHC6', 'ZFP1', 'ZFP42', 'ZFTRAF1', 'ZNF143', 'ZNF282']
Refined Community 30: ['CKAP4', 'DAD1', 'DLX4', 'DNER', 'TYMS']
Refined Community 31: ['AGAP4', 'ATL2', 'ATM', 'CCR2', 'CD86', 'FBXL12', 'FGL2', 'FKBP1A', 'FNTA', 'GIMAP6', 'GLRX', 'HMGXB3', 'ITGA4', 'LZTR1', 'MAFB', 'MIEF1', 'MS4A6A', 'MUSK', 'MYO5A', 'MYO7A', 'NIPSNAP3B', 'PATZ1', 'PPIG', 'PRKACA', 'PRKCSH', 'RUNX1', 'SIRT6', 'TRAT1', 'TRBC1', 'UVRAG']
Refined Community 32: ['ADD1', 'ADGRG6', 'ASB9', 'ATRN', 'CTBP2', 'IFT74', 'LIFR', 'LRPAP1', 'MAEA', 'PREPL', 'SERINC3', 'SLC44A4', 'SLC6A14', 'SORBS2', 'WFDC2', 'YWHAB', 'ZNF140']
Refined Community 33: ['AKT1', 'AKT2', 'AKT3', 'ALK', 'ARAF', 'BAD', 'BAK1', 'BAX', 'BID', 'BRAF', 'CASP3', 'CASP8', 'CASP9', 'CCND1', 'CDK4', 'CDK6', 'CDKN1A', 'CDKN2A', 'CRABP1', 'CRABP2', 'CYCS', 'DDB2', 'E2F1', 'E2F2', 'E2F3', 'EGF', 'EGFR', 'EML4', 'ERBB2', 'FHIT', 'FOXO3', 'GADD45A', 'GADD45B', 'GADD45G', 'GRB2', 'HRAS', 'JAK3', 'KRAS', 'MAP2K1', 'MAP2K2', 'MAPK1', 'MAPK3', 'NRAS', 'PDK1', 'PIK3CA', 'PIK3CB', 'PIK3CD', 'PIK3R1', 'PIK3R2', 'PIK3R3', 'PLCG1', 'PLCG2', 'POLK', 'PRKCA', 'PRKCB', 'PRKCG', 'RAF1', 'RARB', 'RASSF1', 'RASSF5', 'RB1', 'RXRA', 'RXRB', 'RXRG', 'SOS1', 'SOS2', 'STAT3', 'STAT5A', 'STAT5B', 'STK4', 'TGFA', 'TP53']
Refined Community 34: ['AKT1', 'AKT2', 'AKT3', 'APAF1', 'BAK1', 'BAX', 'BCL2', 'BCL2L1', 'BID', 'BIRC2', 'BIRC3', 'BIRC7', 'BIRC8', 'CASP3', 'CASP8', 'CASP9', 'CCND1', 'CCNE1', 'CCNE2', 'CDK2', 'CDK4', 'CDK6', 'CDKN1A', 'CDKN1B', 'CDKN1C', 'CDKN2B', 'CHUK', 'CKS1B', 'CKS2', 'COL4A1', 'COL4A2', 'COL4A3', 'COL4A4', 'COL4A5', 'COL4A6', 'CYCS', 'DDB2', 'E2F1', 'E2F2', 'E2F3', 'FHIT', 'FN1', 'GADD45A', 'GADD45B', 'GADD45G', 'IKBKB', 'IKBKG', 'ITGA2', 'ITGA2B', 'ITGA3', 'ITGA6', 'ITGAV', 'ITGB1', 'LAMA1', 'LAMA2', 'LAMA3', 'LAMA4', 'LAMA5', 'LAMB1', 'LAMB2', 'LAMB3', 'LAMB4', 'LAMC1', 'LAMC2', 'LAMC3', 'MAX', 'MYC', 'NFKB1', 'NFKBIA', 'NFKBIB', 'NOS2', 'PIK3CA', 'PIK3CB', 'PIK3CD', 'PIK3R1', 'PIK3R2', 'PIK3R3', 'POLK', 'PTEN', 'PTGS2', 'PTK2', 'RARB', 'RB1', 'RELA', 'RXRA', 'RXRB', 'RXRG', 'SKP1', 'TP53', 'TRAF1', 'TRAF2', 'TRAF3', 'TRAF4', 'TRAF5', 'TRAF6', 'ZBTB17']
Refined Community 35: ['ACYP1', 'ATP1A2', 'FAU', 'H2AC8', 'RPL26L1', 'RPL39', 'S100A6', 'TMSB10', 'TMSB4X']
Refined Community 36: ['A2M', 'ACTB', 'ACTC1', 'AHSG', 'ALAD', 'ALB', 'ALDH3A1', 'ALDOA', 'ANXA3', 'ARHGDIA', 'BGN', 'C2', 'CDH1', 'CFB', 'CFL2', 'CLSTN1', 'CLU', 'COL18A1', 'COL1A1', 'CP', 'CTSA', 'CTSD', 'CTSV', 'CTSZ', 'CX3CL1', 'EEF1A2', 'EEF2', 'ENO1', 'FN1', 'GAPDH', 'GLO1', 'GOT1', 'HMGB1', 'IGFBP4', 'IGFBP7', 'KCTD7', 'LDHA', 'LGALS3BP', 'MDH2', 'PCNA', 'PEBP1', 'PHACTR4', 'PPIB', 'PRDX1', 'PZP', 'RAN', 'RPL10A', 'SDC4', 'SERPINB6', 'SERPINC1', 'SPP1', 'SRSF2', 'TF', 'TKT', 'TNXB', 'TPI1', 'TPM3', 'TPT1', 'TUBA1C', 'TUBB', 'VCAM1', 'VCL', 'VCP', 'VIM', 'YWHAE', 'YWHAG', 'YWHAZ']
Refined Community 37: ['ACTA2', 'ACTB', 'ACTN1', 'ACTN4', 'AGRN', 'AHSG', 'AKR1B1', 'ALDOA', 'ANXA4', 'APOH', 'APP', 'ARHGDIA', 'ATP6AP2', 'BGN', 'C2', 'CALR', 'CANT1', 'CAPG', 'CD81', 'CD9', 'CDH1', 'CDH17', 'CFB', 'CHAF1A', 'CLSTN1', 'CLU', 'COL18A1', 'COL1A1', 'CP', 'CST3', 'CTRB1', 'CTSB', 'CTSD', 'CTSZ', 'CX3CL1', 'CYTIP', 'DNPEP', 'ECM1', 'EEF1A2', 'EEF2', 'ENO1', 'ENO3', 'ENPP2', 'FAM3C', 'FN1', 'FUCA1', 'GLO1', 'GLOD4', 'GOT1', 'GPI', 'GSR', 'GSTM5', 'GSTO1', 'GSTP1', 'HGFAC', 'HMGB1', 'HPRT1', 'HSP90AB1', 'HSP90B1', 'IMPA1', 'ITIH2', 'KCTD7', 'KRT73', 'KXD1', 'LAP3', 'LDHA', 'LGALS3BP', 'LGALS4', 'LMNA', 'LXN', 'MDH2', 'ME1', 'MSLN', 'MSN', 'MTAP', 'NPC2', 'NPM1', 'NUCB1', 'P4HB', 'PARK7', 'PEBP1', 'PGAM2', 'PGK1', 'PHACTR4', 'PKM', 'PNP', 'PPIC', 'PRDX1', 'PRDX6', 'PRSS2', 'PSMA1', 'PSMA3', 'PSMA4', 'PSMA5', 'PSMA7', 'PSMB1', 'PSMB2', 'PSMB3', 'PSMB4', 'PSMB5', 'PSMB6', 'QSOX1', 'RACK1', 'RAN', 'RHOA', 'RPL10A', 'RPL18A', 'SDC4', 'SEMA3C', 'SERPINB6', 'SERPINC1', 'SET', 'SLK', 'SPARC', 'SPP1', 'TF', 'TIMP2', 'TINAGL1', 'TNXB', 'TPI1', 'TPT1', 'UBA52', 'VCAM1', 'VCL', 'VCP', 'WDR1', 'YWHAB', 'YWHAE', 'YWHAH', 'YWHAQ', 'YWHAZ']
Refined Community 38: ['ACTB', 'AGRN', 'AKR1A1', 'ALDOA', 'ANXA2', 'APP', 'BGN', 'C2', 'CANT1', 'CD81', 'CDC25B', 'CDH1', 'CDH17', 'CFB', 'CHAF1A', 'CLSTN1', 'CLU', 'COL18A1', 'CP', 'CREG1', 'CST3', 'CTSA', 'CTSB', 'CTSD', 'CTSH', 'CTSV', 'CX3CL1', 'CYTIP', 'DHFR', 'DUT', 'ENO1', 'ENPP2', 'FDPS', 'FN1', 'GOT1', 'GOT2', 'GPRASP1', 'GSTM5', 'GSTO1', 'HSP90AB1', 'IGFBP4', 'ITIH2', 'KCTD7', 'LDHA', 'LGALS3BP', 'MDH1', 'MDH2', 'MSLN', 'NPC2', 'PEBP1', 'PGAM2', 'PGK1', 'PHACTR4', 'PPIB', 'PRDX1', 'PRSS2', 'PSAT1', 'PSMA4', 'PSMA6', 'PSMA7', 'PSMB5', 'PSMB6', 'PSMB7', 'RACK1', 'RALYL', 'RAN', 'RPL10A', 'SDC1', 'SDC4', 'TALDO1', 'TIMP2', 'TPI1', 'TPT1', 'TUT1', 'VCAM1', 'YWHAZ']
"""    
    detailed_df = run_optimized_analysis(
        input_text=input_text,
        api_key=os.environ.get("DEEPSEEK_API_KEY"), 
        detailed_csv="Lung Cancer Annotation.csv"
    )
    
    pd.set_option('display.max_colwidth', None)
    print("\nPathway Analysis Results (Clean Text):")

    main_columns = ['Community', 'Process_With_Enrichment', 'Confidence_With_Enrichment', 
                   'Process_Without_Enrichment', 'Confidence_Without_Enrichment', 
                   'Final_Process', 'Final_Confidence']
    available_main_columns = [col for col in main_columns if col in detailed_df.columns]
    print(detailed_df[available_main_columns])
    
    print(f"\nTotal communities analyzed: {len(detailed_df)}")
    print(f"Available columns: {list(detailed_df.columns)}")
    
    if 'Contributing_Genes_With_Enrichment' in detailed_df.columns:
        print(f"\nSample contributing genes (cleaned):")
        for idx, row in detailed_df.head(3).iterrows():
            print(f"Community {row['Community']}: {row.get('Contributing_Genes_With_Enrichment', 'N/A')}")

### Clean Process Name Columns — AML Annotation

Removes trailing embedded confidence-score artifacts from `Process_With_Enrichment` and
`Process_Without_Enrichment`, e.g.:

- `Name ([0.42])`
- `Name (0.68)`
- `Name (confidence score0.65)`
- `Name (confidence score: 0.65)`

Genuine parenthetical content in the middle of a name (e.g. `... (SREBP) Signaling`) is left alone,
since the pattern only matches a trailing parenthetical block that contains a number.

In [ ]:
COLUMNS_TO_CLEAN = [
    "Process_With_Enrichment",
    "Process_Without_Enrichment",
]

SCORE_SUFFIX_PATTERN = re.compile(
    r"""
    \s*                                  # leading whitespace before the paren block
    \(+                                  # one or more opening parens
    \s*
    (?:confidence\s*score\s*:?\s*)?      # optional "confidence score" / "confidence score:" label
    \[?\s*                               # optional opening bracket
    (-?\d*\.\d+|-?\d+)                   # the numeric score itself
    \s*\]?                               # optional closing bracket
    \s*
    \)+                                  # one or more closing parens
    \s*$                                 # anchored to end of string
    """,
    re.IGNORECASE | re.VERBOSE,
)
def clean_process_name(value, extract_score: bool = False):
    if not isinstance(value, str):
        return (value, None) if extract_score else value

    match = SCORE_SUFFIX_PATTERN.search(value)
    cleaned = SCORE_SUFFIX_PATTERN.sub("", value).strip()

    if extract_score:
        score = float(match.group(1)) if match else None
        return cleaned, score
    return cleaned
def clean_annotation_file(input_path: str, output_path: str, extract_scores: bool = True):
    df = pd.read_csv(input_path)
    df.columns = [c.strip() for c in df.columns]

    for col in COLUMNS_TO_CLEAN:
        if col not in df.columns:
            print(f"  Skipping \'{col}\' - column not found")
            continue

        if extract_scores:
            cleaned_and_scores = df[col].apply(lambda v: clean_process_name(v, extract_score=True))
            df[col] = cleaned_and_scores.apply(lambda t: t[0])
            score_col = f"{col}_Score_Embedded"
            df[score_col] = cleaned_and_scores.apply(lambda t: t[1])
            n_found = df[score_col].notna().sum()
            print(f"  {col}: cleaned {n_found} embedded score(s) -> new column \'{score_col}\'")
        else:
            df[col] = df[col].apply(clean_process_name)
            print(f"  {col}: cleaned")

    df.to_csv(output_path, index=False)
    print(f"Saved cleaned file to {output_path}")
    return df
INPUT_CSV = "Lung Cancer Annotation.csv"
OUTPUT_CSV = "Lung Cancer Annotation Cleaned.csv"

df = clean_annotation_file(INPUT_CSV, OUTPUT_CSV)
df.head()

### Validation Pipeline

In [ ]:
import pandas as pd
import json
import os
import re
import time
import requests
from typing import List, Dict, Any, Optional, Tuple
from tqdm import tqdm

UNKNOWN_PROCESS_LABEL = "unknown process"

def _is_unknown(process_name: str) -> bool:
    """Return True if the process name is a variant of 'unknown process'."""
    return process_name.strip().lower() == UNKNOWN_PROCESS_LABEL


class GeneSetValidator:
    def __init__(
        self,
        gene_sets_path: str,
        papers_csv: str = "AML_Paper_DB.csv",
        api_key: Optional[str] = None,
        output_path: str = "validated_gene_sets.csv",
    ):
        self.gene_sets_path = gene_sets_path
        self.papers_csv     = papers_csv
        self.output_path    = output_path
        self.api_key        = api_key or os.environ.get("DEEPSEEK_API_KEY")

        if not self.api_key:
            print("Warning: No DeepSeek API key provided.")

        self.gene_sets_df      = None
        self.papers_df         = None
        self.validated_results = []

        # ── Load Clarivate JIF journal list ───────────────────────────────────
        try:
            jif_df = pd.read_csv("journals_filtered_JIF_ge_4.csv", sep=",")
            jif_df.columns = [c.strip() for c in jif_df.columns]

            self._hq_issn_set  = set()
            self._hq_eissn_set = set()
            self.TOP_JOURNALS  = []

            for _, jr in jif_df.iterrows():
                issn  = str(jr.get("ISSN",  "")).strip().replace("-", "").upper()
                eissn = str(jr.get("eISSN", "")).strip().replace("-", "").upper()
                name  = str(jr.get("Journal name", "")).strip()

                if issn  and issn  != "NAN": self._hq_issn_set.add(issn)
                if eissn and eissn != "NAN": self._hq_eissn_set.add(eissn)
                if name:                     self.TOP_JOURNALS.append(name)

            self._top_journal_set_normalized = {
                self.normalize_journal(j) for j in self.TOP_JOURNALS
            }

            print(f"Loaded {len(jif_df)} high-quality journals from Clarivate JIF CSV "
                  f"({len(self._hq_issn_set)} ISSNs, {len(self._hq_eissn_set)} eISSNs)")

        except FileNotFoundError:
            print("Warning: journals_filtered_JIF_ge_4.csv not found. No quality filter applied.")
            self._hq_issn_set                = set()
            self._hq_eissn_set               = set()
            self.TOP_JOURNALS                = []
            self._top_journal_set_normalized = set()


    def normalize_journal(self, name) -> str:
        if pd.isna(name):
            return ""
        name = str(name).lower()
        name = name.split(" : ")[0].split(" - ")[0]
        name = re.sub(r"[^\w\s]", "", name)
        name = re.sub(r"\s+", " ", name)
        return name.strip()

    def _normalize_id(self, val) -> str:
        return str(val).strip().replace("-", "").upper() if pd.notna(val) else ""

    def _is_high_quality(self, row) -> bool:
        """Match in priority order: ISSN -> eISSN -> exact normalized journal name only."""
        issn = self._normalize_id(row.get("issn"))
        if issn and issn in self._hq_issn_set:
            return True
        eissn = self._normalize_id(row.get("eissn"))
        if eissn and eissn in self._hq_eissn_set:
            return True
        norm = self.normalize_journal(row.get("journal", ""))
        if norm and norm in self._top_journal_set_normalized:
            return True
        return False

    def _load_papers_for_genes(self, genes: List[str]) -> pd.DataFrame:
        if self.papers_df is None or self.papers_df.empty:
            return pd.DataFrame()

        genes_upper = {g.upper() for g in genes if g}
        mask   = self.papers_df["gene"].str.upper().isin(genes_upper)
        subset = self.papers_df[mask].copy()

        if subset.empty:
            return subset

        if "full_text" not in subset.columns:
            subset["full_text"] = "Full text not available via PMC"

        subset["matched_journal"] = subset.apply(
            lambda row: "HQ" if self._is_high_quality(row) else None, axis=1
        )

        hq  = subset["matched_journal"].notna().sum()
        tot = len(subset)
        print(f"  Paper filter: {hq}/{tot} high-quality papers for gene set")
        return subset


    def load_data(self):
        print("Loading gene sets data...")
        self.gene_sets_df = pd.read_csv(self.gene_sets_path)
        self.gene_sets_df.columns = [col.strip() for col in self.gene_sets_df.columns]
        print(f"Loaded {len(self.gene_sets_df)} gene sets from {self.gene_sets_path}")

        print(f"Loading AML paper database from {self.papers_csv}...")
        if not os.path.exists(self.papers_csv):
            print(f"Paper CSV '{self.papers_csv}' not found.")
            self.papers_df = pd.DataFrame()
        else:
            self.papers_df = pd.read_csv(self.papers_csv, low_memory=False)
            self.papers_df.columns = [col.strip() for col in self.papers_df.columns]
            if "gene" not in self.papers_df.columns:
                print("'gene' column not found in paper CSV - gene filtering disabled.")
                self.papers_df = pd.DataFrame()
            else:
                self.papers_df["gene"] = self.papers_df["gene"].astype(str).str.strip()
                print(f"Loaded {len(self.papers_df)} papers ({self.papers_df['gene'].nunique()} unique genes)")


    def extract_genes_from_set(self, gene_set_row: pd.Series) -> List[str]:
        genes = []
        possible_gene_columns = [
            "Genes_String", "Contributing_Genes", "Genes", "Final_Contributing_Genes",
        ]
        for col in possible_gene_columns:
            if col in gene_set_row and pd.notna(gene_set_row[col]):
                gene_data = gene_set_row[col]
                if isinstance(gene_data, list):
                    genes = gene_data
                elif isinstance(gene_data, str):
                    if "," in gene_data:
                        genes = [g.strip() for g in gene_data.split(",")]
                    elif ";" in gene_data:
                        genes = [g.strip() for g in gene_data.split(";")]
                    else:
                        genes = [gene_data.strip()]
                break
        return [g for g in genes if g and g.strip()]


    def extract_gene_related_abstracts(
        self,
        genes: List[str],
        papers_df: pd.DataFrame,
        max_papers: int = 50,
    ) -> Tuple[List[Dict], List[str]]:
        """
        Filter to HQ papers only, then prioritise by gene_count descending.

        gene_count = number of query genes found in the COMBINED
        abstract + full_text corpus (never short-circuited by the gene column).
        The gene column is used only as a safety net to ensure the primary
        gene is never missed if the text search fails.

        """
        if isinstance(genes, str):
            genes = [g.strip() for g in genes.split(",")]

        genes_set = {g.upper() for g in genes if g}

        hq_papers = []

        for _, paper in papers_df.iterrows():

            if pd.isna(paper.get("matched_journal")):
                continue

            abstract  = str(paper.get("abstract",  ""))
            full_text = str(paper.get("full_text", ""))

            if abstract  in ("nan", "Abstract not available", ""):
                abstract = ""
            if full_text in ("nan", "Full text not available via PMC", ""):
                full_text = ""

            if not abstract and not full_text:
                continue

            search_corpus = (abstract + " " + full_text).strip()
            mentioned_genes = [
                g for g in genes
                if g and re.search(rf"\b{re.escape(g)}\b", search_corpus, re.IGNORECASE)
            ]
            gene_from_col = str(paper.get("gene", "")).strip()
            if gene_from_col and gene_from_col.upper() in genes_set:
                if gene_from_col not in mentioned_genes:
                    mentioned_genes.append(gene_from_col)

            if not mentioned_genes:
                continue

            truncated_abstract  = (abstract[:500]  + "...") if len(abstract)  > 500  else abstract
            truncated_full_text = (full_text[:3000] + "...") if len(full_text) > 3000 else full_text

            hq_papers.append({
                "title":           paper.get("title",   "No title"),
                "authors":         paper.get("authors", "No authors"),
                "journal":         paper.get("journal", "No journal"),
                "year":            paper.get("year",    "Unknown"),
                "abstract":        truncated_abstract,
                "full_text":       truncated_full_text,
                "doi":             paper.get("doi",     "No DOI"),
                "pmid":            paper.get("pmid",    "No PMID"),
                "genes_mentioned": mentioned_genes,
                "gene_count":      len(mentioned_genes),
            })

        hq_papers.sort(key=lambda x: x["gene_count"], reverse=True)
        relevant_papers = hq_papers[:max_papers]

        top_journal_papers_used = [
            f"Paper {idx}: '{p['title']}', {p['authors']}"
            for idx, p in enumerate(relevant_papers, 1)
        ]

        return relevant_papers, top_journal_papers_used

    def create_validation_prompt(
        self,
        gene_set_id: str,
        genes: List[str],
        process_with_enrichment: str,
        confidence_with_enrichment: float,
        process_without_enrichment: str,
        confidence_without_enrichment: float,
        analysis_with_enrichment: str,
        analysis_without_enrichment: str,
        relevant_papers: List[Dict],
    ) -> str:

        genes_str      = ", ".join(genes) if isinstance(genes, list) else str(genes)
        papers_section = ""
        for i, paper in enumerate(relevant_papers, 1):
            papers_section += (
                f"PAPER {i}:\n"
                f"Title: {paper['title']}\n"
                f"Authors: {paper['authors']}\n"
                f"Journal: {paper['journal']} ({paper['year']})\n"
                f"Genes Mentioned: {', '.join(paper['genes_mentioned'])}\n"
                f"Abstract: {paper['abstract']}\n"
                f"\n"
            )

        return f"""You are a scientific expert in genomics and bioinformatics tasked with validating gene set analysis results using ONLY the provided literature.

GENE SET ID: {gene_set_id}
GENES: {genes_str}

ORIGINAL ANALYSIS WITH ENRICHMENT:
Process Name: {process_with_enrichment}
Original Confidence Score: {confidence_with_enrichment}
Analysis: {analysis_with_enrichment[:1000]}...

ORIGINAL ANALYSIS WITHOUT ENRICHMENT:
Process Name: {process_without_enrichment}
Original Confidence Score: {confidence_without_enrichment}
Analysis: {analysis_without_enrichment[:1000]}...

PROVIDED LITERATURE (USE ONLY THESE STUDIES):
{papers_section}

VALIDATION TASK:
Based STRICTLY on the provided literature above, evaluate both analyses and provide updated confidence scores.

**ABSOLUTE REQUIREMENT FOR PAPER CITATIONS:**
Whenever you reference a paper anywhere in your response, always use this exact format:
"Paper X (FirstAuthorLastName et al.)"
For example: "Paper 3 (Nakamura et al.) shows..." or "supported by Paper 1 (Chen et al.) and Paper 4 (Okafor et al.)"
Do NOT use bare numbers like "Paper 3" alone, and do NOT spell out full titles or complete author lists in-line — the first-author-et-al form is sufficient everywhere in the response.

CRITICAL REQUIREMENTS:
1. Use ONLY the provided papers - do not add external knowledge
2. For each process, check if the genes are supported by the literature
3. Provide updated confidence scores based on evidence strength
4. Select the better-supported process as the final choice
5. **MANDATORY:  When referencing papers by number anywhere in your analysis text, always use the "Paper X (FirstAuthor et al.)" format defined above — never bare numbers, never full citations..**
6. The final selected process MUST be either:
   (a) exactly one of the two original process names, OR
   (b) "Neither process" ONLY if both updated confidence scores (Updated Confidence With Enrichment, Updated Confidence Without Enrichment) are <= 0.05.

7. The final confidence MUST follow:
   - If a process is selected: Final Confidence = the updated confidence of that selected process.
   - If "Neither process" is selected: Final Confidence = max(Updated Confidence With Enrichment, Updated Confidence Without Enrichment).

FORMAT YOUR RESPONSE EXACTLY AS FOLLOWS:

VALIDATION OF ENRICHMENT ANALYSIS:
Evidence Assessment: [Detailed assessment based strictly on provided papers]
Original Confidence: {confidence_with_enrichment}
Updated Confidence: [Your revised score 0.00-1.00 based on literature evidence]
Supporting Papers: [List paper numbers that support this process]

VALIDATION OF DIRECT ANALYSIS:
Evidence Assessment: [Detailed assessment based strictly on provided papers]
Original Confidence: {confidence_without_enrichment}
Updated Confidence: [Your revised score 0.00-1.00 based on literature evidence]
Supporting Papers: [List paper numbers that support this process]

FINAL PROCESS SELECTION:
Selected Process: [Choose the better-supported process name]
Final Confidence: [The updated confidence score for your selected process]
Selection Reasoning: [Explain why this process has stronger literature support]

CONFLICT ANALYSIS:
Review all papers for contradictory evidence about gene functions, pathway assignments, or experimental results.
CONFLICTING_EVIDENCE_FOUND: [TRUE/FALSE]
CONFLICT_DESCRIPTION: [Brief description of any conflicts found, or "No conflicts detected"]

SUPPORTING CITATIONS:
[List citations in format: "Title, Authors" for papers that support the final selected process]

VALIDATION ANALYSIS TEXT:
[Comprehensive summary of all changes made, reasoning for confidence adjustments, and evidence from the provided studies that led to the final process selection.
Use "Paper X (FirstAuthor et al.)" format for any paper references.]"""

    def call_deepseek_api(self, prompt: str) -> str:
        if not self.api_key:
            raise ValueError("DeepSeek API key is required.")

        api_url = "https://api.deepseek.com/v1/chat/completions"
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type":  "application/json",
        }
        data = {
            "model": "deepseek-chat",
            "messages": [
                {
                    "role":    "system",
                    "content": (
                        "You are a scientific expert in genomics and bioinformatics "
                        "specializing in gene set analysis validation. "
                        "Base your analysis strictly on the provided literature."
                    ),
                },
                {"role": "user", "content": prompt},
            ],
            "temperature": 0,
            "max_tokens":  8000,
        }

        max_retries = 3
        retry_delay = 5

        for attempt in range(max_retries):
            try:
                response = requests.post(api_url, headers=headers, json=data, timeout=180)

                if response.status_code == 200:
                    return response.json()["choices"][0]["message"]["content"]

                elif response.status_code == 429:
                    wait_time = int(response.headers.get("Retry-After", retry_delay * 2))
                    print(f"Rate limited. Waiting {wait_time}s...")
                    time.sleep(wait_time)

                elif 500 <= response.status_code < 600:
                    print(f"Server error {response.status_code}. Retrying in {retry_delay}s...")
                    time.sleep(retry_delay)
                    retry_delay *= 2

                else:
                    raise Exception(f"API Error {response.status_code}: {response.text}")

            except requests.exceptions.Timeout:
                print(f"Timeout on attempt {attempt + 1}/{max_retries}")
                if attempt < max_retries - 1:
                    time.sleep(retry_delay)
                    retry_delay *= 2
                else:
                    raise Exception(f"API timeout after {max_retries} attempts")

            except requests.exceptions.RequestException as e:
                if attempt < max_retries - 1:
                    time.sleep(retry_delay)
                    retry_delay *= 2
                else:
                    raise Exception(f"Request failed after {max_retries} attempts: {e}")

        raise Exception(f"Failed to get a valid response after {max_retries} attempts")

    def parse_llm_response(self, response: str) -> Dict[str, Any]:
        results = {}

        m = re.search(
            r"VALIDATION OF ENRICHMENT ANALYSIS:.*?Updated Confidence:\s*([\d.]+)",
            response, re.DOTALL,
        )
        if m:
            results["confidence_with_enrichment_after"] = float(m.group(1))

        m = re.search(
            r"VALIDATION OF DIRECT ANALYSIS:.*?Updated Confidence:\s*([\d.]+)",
            response, re.DOTALL,
        )
        if m:
            results["confidence_without_enrichment_after"] = float(m.group(1))

        m = re.search(r"Selected Process:\s*(.+?)(?=\n|$)", response)
        if m:
            results["final_process"] = m.group(1).strip()

        m = re.search(r"Final Confidence:\s*([\d.]+)", response)
        if m:
            results["final_confidence"] = float(m.group(1))

        m = re.search(r"CONFLICTING_EVIDENCE_FOUND:\s*(TRUE|FALSE)", response, re.IGNORECASE)
        results["conflicting_evidence_found"] = (
            m.group(1).upper() == "TRUE" if m else False
        )

        m = re.search(
            r"CONFLICT_DESCRIPTION:\s*(.+?)(?=\n\n|SUPPORTING CITATIONS:|$)",
            response, re.DOTALL,
        )
        results["conflict_description"] = m.group(1).strip() if m else "No conflicts detected"

        m = re.search(
            r"SUPPORTING CITATIONS:\s*(.+?)(?=\n\n|VALIDATION ANALYSIS TEXT:|$)",
            response, re.DOTALL,
        )
        if m:
            results["supporting_citations"] = [
                line.strip()
                for line in m.group(1).strip().split("\n")
                if line.strip() and not line.strip().startswith(("-", "*"))
            ]
        else:
            results["supporting_citations"] = []

        m = re.search(r"VALIDATION ANALYSIS TEXT:\s*(.+?)$", response, re.DOTALL)
        if m:
            results["validation_analysis_text"] = m.group(1).strip()

        return results

    def validate_gene_set(self, gene_set_row: pd.Series) -> Dict[str, Any]:
        set_id = str(
            gene_set_row.get("Community", "")
            or gene_set_row.get("Set_ID", "")
            or gene_set_row.get("community", "")
            or gene_set_row.get("index", "")
        )

        genes = self.extract_genes_from_set(gene_set_row)
        if not genes:
            print(f"  No genes found for set {set_id}")
            return self._error_result(set_id, "No genes found for this set")

        process_with    = str(gene_set_row.get("Process_With_Enrichment", "")
                              or gene_set_row.get("Final_Process", ""))
        conf_with       = float(gene_set_row.get("Confidence_With_Enrichment", 0)
                                or gene_set_row.get("Final_Confidence", 0))
        process_without = str(gene_set_row.get("Process_Without_Enrichment", ""))
        conf_without    = float(gene_set_row.get("Confidence_Without_Enrichment", 0))

        analysis_with    = str(gene_set_row.get("Analysis_Text_With_Enrichment", "")
                               or gene_set_row.get("Pathway_Reasoning_With_Enrichment", "")
                               or gene_set_row.get("Full_Analysis", ""))
        analysis_without = str(gene_set_row.get("Analysis_Text_Without_Enrichment", "")
                               or gene_set_row.get("Pathway_Reasoning_Without_Enrichment", ""))

        # ── Unknown process guard ─────────────────────────────────────────────
        with_is_unknown    = _is_unknown(process_with)
        without_is_unknown = _is_unknown(process_without)

        if with_is_unknown and without_is_unknown:
            print(f"  Both processes are 'unknown process' for set {set_id} - skipping.")
            return self._both_unknown_result(set_id, genes, gene_set_row)

        if with_is_unknown:
            print(f"  'With enrichment' is unknown process for set {set_id} - zeroing out, validating direct only.")
            process_with  = UNKNOWN_PROCESS_LABEL
            conf_with     = 0.0
            analysis_with = "Unknown process - not validated."

        if without_is_unknown:
            print(f"  'Without enrichment' is unknown process for set {set_id} - zeroing out, validating enrichment only.")
            process_without  = UNKNOWN_PROCESS_LABEL
            conf_without     = 0.0
            analysis_without = "Unknown process - not validated."

        papers_df = self._load_papers_for_genes(genes)

        relevant_papers, top_journal_info = self.extract_gene_related_abstracts(
            genes, papers_df, max_papers=50
        )

        if not relevant_papers:
            print(f"  No matching high-quality papers for set {set_id}")
            return self._no_papers_result(set_id, genes, gene_set_row)

        prompt = self.create_validation_prompt(
            gene_set_id=set_id,
            genes=genes,
            process_with_enrichment=process_with,
            confidence_with_enrichment=conf_with,
            process_without_enrichment=process_without,
            confidence_without_enrichment=conf_without,
            analysis_with_enrichment=analysis_with,
            analysis_without_enrichment=analysis_without,
            relevant_papers=relevant_papers,
        )

        os.makedirs("prompts", exist_ok=True)
        with open(f"prompts/prompt_{set_id}.txt", "w", encoding="utf-8") as f:
            f.write(prompt)

        try:
            response = self.call_deepseek_api(prompt)

            os.makedirs("responses", exist_ok=True)
            with open(f"responses/response_{set_id}.txt", "w", encoding="utf-8") as f:
                f.write(response)

            parsed = self.parse_llm_response(response)

            conf_with_after    = parsed.get("confidence_with_enrichment_after",   conf_with)
            conf_without_after = parsed.get("confidence_without_enrichment_after", conf_without)
            if with_is_unknown:
                conf_with_after = 0.0
            if without_is_unknown:
                conf_without_after = 0.0

            return {
                "Set_ID":                                set_id,
                "Genes":                                 genes,
                "Process_With_Enrichment_Original":      process_with,
                "Process_Without_Enrichment_Original":   process_without,
                "Confidence_With_Enrichment_Before":     conf_with,
                "Confidence_Without_Enrichment_Before":  conf_without,
                "Confidence_With_Enrichment_After":      conf_with_after,
                "Confidence_Without_Enrichment_After":   conf_without_after,
                "Final_Process":                         parsed.get("final_process",   process_with),
                "Final_Confidence":                      parsed.get("final_confidence", conf_with),
                "Validation_Analysis_Text":              parsed.get("validation_analysis_text", "No analysis provided"),
                "Supporting_Citations":                  parsed.get("supporting_citations", []),
                "Conflicting_Evidence_Found":            parsed.get("conflicting_evidence_found", False),
                "Conflict_Description":                  parsed.get("conflict_description", "No conflicts detected"),
                "Total_Papers_Found":                    len(relevant_papers),
            }

        except Exception as e:
            print(f"  API error for set {set_id}: {e}")
            return self._error_result(set_id, f"API error: {e}", genes, gene_set_row)

    def validate_all_gene_sets(self):
        if self.gene_sets_df is None:
            self.load_data()

        results    = []
        total_sets = len(self.gene_sets_df)
        print(f"\nStarting validation of {total_sets} gene sets...")

        pbar = tqdm(
            total=total_sets,
            desc="Validating gene sets",
            bar_format="{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]",
            ncols=100,
        )

        for i, (_, row) in enumerate(self.gene_sets_df.iterrows()):
            set_id = str(
                row.get("Community", "")
                or row.get("Set_ID", "")
                or row.get("community", "")
                or i
            )
            try:
                result = self.validate_gene_set(row)
                results.append(result)
            except Exception as e:
                print(f"\nUnexpected error on set {set_id}: {e}")

            pbar.update(1)
            pbar.set_postfix_str(f"Set {set_id}")

            if (i + 1) % 5 == 0:
                self.validated_results = results
                self.save_results(f"{self.output_path}.partial")

        pbar.close()
        self.validated_results = results
        self.save_results(self.output_path)
        print(f"\nValidation complete! Processed {len(results)} gene sets")
        return results

    def save_results(self, output_path: str = None):
        if not self.validated_results:
            print("No results to save")
            return

        path = output_path or self.output_path
        df   = pd.DataFrame(self.validated_results)

        required_columns = [
            "Set_ID", "Genes",
            "Process_With_Enrichment_Original", "Process_Without_Enrichment_Original",
            "Confidence_With_Enrichment_Before", "Confidence_Without_Enrichment_Before",
            "Confidence_With_Enrichment_After",  "Confidence_Without_Enrichment_After",
            "Final_Process", "Final_Confidence",
            "Validation_Analysis_Text", "Supporting_Citations",
            "Conflicting_Evidence_Found", "Conflict_Description",
            "Total_Papers_Found",
        ]
        for col in required_columns:
            if col not in df.columns:
                df[col] = None
        df = df[required_columns]
        df.to_csv(path, index=False)
        print(f"Saved {len(df)} validated results to {path}")


    def generate_summary_report(self) -> str:
        if not self.validated_results:
            return "No validation results available"

        total = len(self.validated_results)

        conflict_count     = sum(1 for r in self.validated_results if r.get("Conflicting_Evidence_Found"))
        total_paper_counts = [r.get("Total_Papers_Found", 0)           for r in self.validated_results]
        final_confidences  = [r.get("Final_Confidence", 0)             for r in self.validated_results]

        conf_delta_enrich = []
        conf_delta_direct = []
        for r in self.validated_results:
            be, ae = r.get("Confidence_With_Enrichment_Before", 0),   r.get("Confidence_With_Enrichment_After", 0)
            bd, ad = r.get("Confidence_Without_Enrichment_Before", 0), r.get("Confidence_Without_Enrichment_After", 0)
            if be > 0 and ae > 0: conf_delta_enrich.append(ae - be)
            if bd > 0 and ad > 0: conf_delta_direct.append(ad - bd)

        lines = [
            "Gene Set Validation Summary Report",
            "=" * 50,
            f"Total gene sets processed: {total}",
            "",
            "CONFLICT ANALYSIS:",
            f"  With conflicts:    {conflict_count} ({conflict_count/total*100:.1f}%)",
            f"  Without conflicts: {total - conflict_count} ({(total-conflict_count)/total*100:.1f}%)",
            "",
            "PAPER DISCOVERY:",
            f"  Avg papers/set:  {sum(total_paper_counts)/len(total_paper_counts):.2f}",
            f"  Sets with papers: {sum(1 for c in total_paper_counts if c > 0)} ({sum(1 for c in total_paper_counts if c > 0)/total*100:.1f}%)",
            "",
            "CONFIDENCE:",
            f"  Avg final confidence: {sum(final_confidences)/len(final_confidences):.2f}",
            f"  Range: {min(final_confidences):.2f} - {max(final_confidences):.2f}",
        ]
        if conf_delta_enrich:
            lines.append(f"  Avg enrichment confidence delta: {sum(conf_delta_enrich)/len(conf_delta_enrich):+.3f}")
        if conf_delta_direct:
            lines.append(f"  Avg direct confidence delta:     {sum(conf_delta_direct)/len(conf_delta_direct):+.3f}")

        return "\n".join(lines)


    def _error_result(
        self,
        set_id: str,
        error_msg: str,
        genes: List[str] = None,
        row: pd.Series = None,
    ) -> Dict[str, Any]:
        genes = genes or []
        return {
            "Set_ID":                                set_id,
            "Genes":                                 genes,
            "Process_With_Enrichment_Original":      row.get("Process_With_Enrichment", "") if row is not None else "",
            "Process_Without_Enrichment_Original":   row.get("Process_Without_Enrichment", "") if row is not None else "",
            "Confidence_With_Enrichment_Before":     row.get("Confidence_With_Enrichment", 0) if row is not None else 0,
            "Confidence_Without_Enrichment_Before":  row.get("Confidence_Without_Enrichment", 0) if row is not None else 0,
            "Confidence_With_Enrichment_After":      0,
            "Confidence_Without_Enrichment_After":   0,
            "Final_Process":                         "ERROR",
            "Final_Confidence":                      0,
            "Validation_Analysis_Text":              error_msg,
            "Supporting_Citations":                  [],
            "Conflicting_Evidence_Found":            False,
            "Conflict_Description":                  "Error occurred during validation",
            "Total_Papers_Found":                    0,
        }

    def _no_papers_result(
        self, set_id: str, genes: List[str], row: pd.Series
    ) -> Dict[str, Any]:
        conf_with = float(row.get("Confidence_With_Enrichment", 0) or row.get("Final_Confidence", 0))
        return {
            "Set_ID":                                set_id,
            "Genes":                                 genes,
            "Process_With_Enrichment_Original":      str(row.get("Process_With_Enrichment", "") or row.get("Final_Process", "")),
            "Process_Without_Enrichment_Original":   str(row.get("Process_Without_Enrichment", "")),
            "Confidence_With_Enrichment_Before":     conf_with,
            "Confidence_Without_Enrichment_Before":  float(row.get("Confidence_Without_Enrichment", 0)),
            "Confidence_With_Enrichment_After":      conf_with,
            "Confidence_Without_Enrichment_After":   float(row.get("Confidence_Without_Enrichment", 0)),
            "Final_Process":                         str(row.get("Process_With_Enrichment", "") or row.get("Final_Process", "")),
            "Final_Confidence":                      conf_with,
            "Validation_Analysis_Text":              "No high-quality papers found for this gene set",
            "Supporting_Citations":                  [],
            "Conflicting_Evidence_Found":            False,
            "Conflict_Description":                  "No high-quality papers available",
            "Total_Papers_Found":                    0,
        }

    def _both_unknown_result(
        self, set_id: str, genes: List[str], row: pd.Series
    ) -> Dict[str, Any]:
        return {
            "Set_ID":                                set_id,
            "Genes":                                 genes,
            "Process_With_Enrichment_Original":      UNKNOWN_PROCESS_LABEL,
            "Process_Without_Enrichment_Original":   UNKNOWN_PROCESS_LABEL,
            "Confidence_With_Enrichment_Before":     0,
            "Confidence_Without_Enrichment_Before":  0,
            "Confidence_With_Enrichment_After":      0,
            "Confidence_Without_Enrichment_After":   0,
            "Final_Process":                         UNKNOWN_PROCESS_LABEL,
            "Final_Confidence":                      0,
            "Validation_Analysis_Text":              "Both processes are unknown - validation skipped.",
            "Supporting_Citations":                  [],
            "Conflicting_Evidence_Found":            False,
            "Conflict_Description":                  "Both processes unknown - no validation performed",
            "Total_Papers_Found":                    0,
        }




# ── Entry point ───────────────────────────────────────────────────────────────

def main():
    gene_sets_path = "Lung Cancer Annotation Cleaned.csv"
    papers_csv     = "LC_Paper_DB.csv"
    output_path    = "Lung Cancer Validation.csv"

    print("AML Gene Set Validation Pipeline")
    print("=" * 50)

    api_key = os.environ.get("DEEPSEEK_API_KEY")
    if not api_key:
        print("No DEEPSEEK_API_KEY env var found.")
        print("Set it with:  export DEEPSEEK_API_KEY=your_key")
        return

    print("API key found")

    validator = GeneSetValidator(
        gene_sets_path=gene_sets_path,
        papers_csv=papers_csv,
        api_key=api_key,
        output_path=output_path,
    )

    validator.load_data()

    test_mode = False

    if test_mode:
        print("\nTest mode - validating first gene set only")
        result = validator.validate_gene_set(validator.gene_sets_df.iloc[0])
        validator.validated_results = [result]
        validator.save_results("test_validation.csv")
        for k, v in result.items():
            print(f"  {k}: {len(v) if isinstance(v, list) else v}")
    else:
        validator.validate_all_gene_sets()

    report = validator.generate_summary_report()
    print("\n" + report)
    with open("validation_summary.txt", "w") as f:
        f.write(report)

    print("\nDone!")


if __name__ == "__main__":
    main() 

In [ ]:
import pandas as pd

path = "Lung Cancer Validation.csv"
df = pd.read_csv(path)

df.loc[df["Final_Confidence"] <= 0.05, "Final_Process"] = "Neither process"

df.to_csv(path, index=False)